# 🧬 Rec-Prep | Receptor Preparation & Grid Map Generation

This notebook automates the complete **rigid (+ optional flexible) receptor preparation pipeline** for AutoDock-GPU, from raw PDB to a full set of AutoGrid4 map files — ready for virtual screening.

---

### 🔬 Pipeline Overview
| Step | Task |
|---|---|
| 1 | Install AGFR Suite + visualization libraries |
| 2 | Upload PDB structure |
| 3 | Prepare receptor → `.pdbqt` |
| 3.5 | *(Optional)* Define flexible residues |
| 4 | Visualize protein in 3D |
| 5 | Detect binding pockets |
| 6 | Define grid box (6 methods: AutoSite pocket / Blind docking / Manual coordinates / Around ligand / Around Cofactors / Around residues) |
| 7 | Generate AutoGrid4 maps |
| 8 | Download output |

---

> 🔖 **Single-name convention:** the `receptor_name` you set in Step 2 is the
> **only** identifier used for every output in this notebook — the prepared
> receptor `.pdbqt`, the optional Step 3.5 rigid/flex split, the `.gpf`/`.glg`,
> and every AutoGrid4 `.map` file. There is no separate `protein_name` to keep
> in sync — one name, zero file-flagging conflicts at docking time.

> ⚠️ **Runtime:** CPU is sufficient for this notebook. GPU is not required.


---
## 🔧 Step 0 — Setup Conda (Colab only)
Installs `condacolab` to enable `conda` on Google Colab.
> ⚠️ **The runtime will restart automatically after this cell.** That is expected — run it once, then continue from Step 1 after the restart.

---

In [ ]:
#@title ▶ Setup Conda on Colab
#@markdown Run this cell **once**. The runtime will restart automatically.
#@markdown After restart, skip this cell and run from **Step 1** onwards.

!pip install -q condacolab
import condacolab
condacolab.install()  # runtime restarts automatically after this

---
## ⚙️ Step 1 — Install Dependencies
Installs the components required for the full pipeline.

> ⏳ This step takes **2–4 minutes**. Run it once per session.


In [ ]:
#@title ▶ Setup Conda + Install ADFRsuite + OpenBabel + PDB2PQR + py3Dmol + AutoDockTools_py3
#@markdown Installs all pipeline dependencies.
#@markdown > ⏳ This step takes **2–4 minutes**. Run it once per session.

import os, subprocess, sys

def _run(cmd, **kw):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)

# ── 1a. py3Dmol ───────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py3Dmol'], check=True)
print('✅ py3Dmol installed')

# ── 1b. ADFRsuite via conda (hcc channel) ─────────────────────────────────────
print('\n📦 Installing adfr-suite via conda (hcc channel) ...')
result = subprocess.run(
    ['conda', 'install', '-y', '-q', 'hcc::adfr-suite'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ adfr-suite installed via conda')
else:
    print('⚠️  conda install output:')
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])
    raise RuntimeError('conda install hcc::adfr-suite failed. Check the output above.')

# ── Update PATH to include conda env bin / ADFRsuite's own bin ───────────────
conda_bin_candidates = [
    '/usr/local/bin',
    '/opt/conda/bin',
    os.path.expanduser('~/anaconda3/bin'),
    os.path.expanduser('~/miniconda3/bin'),
    os.path.expanduser('~/miniconda/bin'),
]
for p in conda_bin_candidates:
    if os.path.isdir(p) and p not in os.environ['PATH']:
        os.environ['PATH'] = p + ':' + os.environ['PATH']

# ── Verify ADFRsuite compiled binaries ────────────────────────────────────────
print('\n🔍 Verifying ADFRsuite binaries ...')
for binary in ['autosite', 'autogrid4']:
    r = subprocess.run(['which', binary], capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '❌'
    print(f'{status} {binary}: {r.stdout.strip() or "NOT FOUND"}')

# ── Locate prepare_receptor4.py and prepare_ligand4.py (ADFRsuite's own) ──────
# These are AutoDockTools scripts bundled INSIDE ADFRsuite, with a shebang into
# ADFRsuite's own self-contained interpreter. They run directly as executables —
# no separate pythonsh / MGLTools install needed, which is exactly what removes
# the Python-2.7-vs-Colab-Python-3 conflict that used to crash the runtime.
def _locate_adfr_script(name):
    # Priority 1: the well-known ADFRsuite bin location
    candidate = f'/usr/local/bin/{name}'
    if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
        return candidate
    # Priority 2: `which` (covers conda env bin, etc.)
    r = subprocess.run(['which', name], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        return r.stdout.strip()
    # Priority 3: broad filesystem search as last resort
    r_find = subprocess.run(
        f'find /usr/local /opt/conda {os.path.expanduser("~")} '
        f'-name "{name}" -maxdepth 10 2>/dev/null | head -1',
        shell=True, capture_output=True, text=True
    )
    found = r_find.stdout.strip()
    return found if found else None

PREPARE_RECEPTOR4 = _locate_adfr_script('prepare_receptor4.py')
PREPARE_LIGAND4   = _locate_adfr_script('prepare_ligand4.py')

print(f'✅ prepare_receptor4.py : {PREPARE_RECEPTOR4}' if PREPARE_RECEPTOR4
      else '❌ prepare_receptor4.py NOT found!')
print(f'✅ prepare_ligand4.py   : {PREPARE_LIGAND4}' if PREPARE_LIGAND4
      else '❌ prepare_ligand4.py NOT found!')

if not PREPARE_RECEPTOR4 or not PREPARE_LIGAND4:
    raise RuntimeError(
        '❌ ADFRsuite AutoDockTools scripts missing.\n'
        '   Ensure the hcc::adfr-suite conda install above completed, then re-run this cell.'
    )

# ── Install prepare_flexreceptor4.py via AutoDockTools_py3 (GitHub) ──────────
# ADFRsuite's conda package does NOT bundle prepare_flexreceptor4.py — only
# prepare_receptor4.py and prepare_ligand4.py ship in its CCSBpckgs tree. The
# flexible-residue split tool only exists in the community Python-3 port of
# classic AutoDockTools, maintained at:
#   https://github.com/Valdes-Tresanco-MS/AutoDockTools_py3
# That repo's setup.py registers 'prepare_flexreceptor4' as a console_scripts
# entry point (no .py suffix), backed by the SAME MolKit/AutoDockTools modules
# already used elsewhere in this pipeline for PDBQT parsing — so this is an
# additive install, not a second toolchain.
#
# This is OPTIONAL: if the clone/install fails (e.g. no network egress to
# github.com in a locked-down environment), Step 3.5 simply reports itself
# unavailable and the pipeline continues as rigid-only — nothing else here
# depends on it.
print('\n📦 Installing prepare_flexreceptor4.py (AutoDockTools_py3 from GitHub) ...')

ADT_PY3_DIR = '/content/AutoDockTools_py3'
PREPARE_FLEXRECEPTOR4 = None

try:
    if not os.path.isdir(ADT_PY3_DIR):
        r_clone = subprocess.run(
            ['git', 'clone', '--quiet', '--depth', '1',
             'https://github.com/Valdes-Tresanco-MS/AutoDockTools_py3.git',
             ADT_PY3_DIR],
            capture_output=True, text=True
        )
        if r_clone.returncode != 0:
            raise RuntimeError(f'git clone failed:\n{r_clone.stderr[-800:]}')
        print(f'   ✅ Cloned AutoDockTools_py3 → {ADT_PY3_DIR}')
    else:
        print(f'   ℹ️  AutoDockTools_py3 already present at {ADT_PY3_DIR} — skipping clone.')

    r_pip = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-e', ADT_PY3_DIR],
        capture_output=True, text=True
    )
    if r_pip.returncode != 0:
        raise RuntimeError(f'pip install -e failed:\n{r_pip.stderr[-800:]}')
    print('   ✅ pip install -e AutoDockTools_py3 completed')

    # The pip console-script wrapper lands in the same bin dir as the running
    # interpreter (e.g. /usr/local/bin under Colab's conda env) — `which`
    # picks it straight off PATH, no .py suffix since it's a generated entry
    # point, not the raw repo file.
    r_which = subprocess.run(['which', 'prepare_flexreceptor4'], capture_output=True, text=True)
    if r_which.returncode == 0 and r_which.stdout.strip():
        PREPARE_FLEXRECEPTOR4 = r_which.stdout.strip()
    else:
        # Fallback: pip sometimes installs into a user-site bin not yet on PATH
        import site
        for bin_dir in (os.path.join(sys.prefix, 'bin'), os.path.join(site.USER_BASE, 'bin')):
            candidate = os.path.join(bin_dir, 'prepare_flexreceptor4')
            if os.path.isfile(candidate):
                if bin_dir not in os.environ['PATH']:
                    os.environ['PATH'] = bin_dir + ':' + os.environ['PATH']
                PREPARE_FLEXRECEPTOR4 = candidate
                break

    if PREPARE_FLEXRECEPTOR4 and not os.access(PREPARE_FLEXRECEPTOR4, os.X_OK):
        os.chmod(PREPARE_FLEXRECEPTOR4, 0o755)

except Exception as e:
    print(f'   ⚠️  AutoDockTools_py3 install failed: {e}')

print(f'✅ prepare_flexreceptor4 : {PREPARE_FLEXRECEPTOR4}' if PREPARE_FLEXRECEPTOR4
      else 'ℹ️  prepare_flexreceptor4 not found — optional Step 3.5 (flexible residues) will be unavailable.')

# Ensure all located scripts are executable (conda extraction occasionally drops +x)
for _script in (PREPARE_RECEPTOR4, PREPARE_LIGAND4, PREPARE_FLEXRECEPTOR4):
    if not _script:
        continue
    if not os.access(_script, os.X_OK):
        os.chmod(_script, 0o755)

# ── Locate AD4.1_bound.dat ────────────────────────────────────────────────────
r = subprocess.run(
    ['find', '/opt/conda', '/usr/local', os.path.expanduser('~'),
     '-name', 'AD4.1_bound.dat', '-maxdepth', '10'],
    capture_output=True, text=True
)
dat_path = r.stdout.strip().split('\n')[0]
if dat_path:
    os.environ['AD4_DAT'] = dat_path
    print(f'✅ AD4.1_bound.dat found: {dat_path}')
else:
    print('⚠️  AD4.1_bound.dat not found — autogrid4 may fail.')

# ── 1c. OpenBabel — three strategies in priority order ────────────────────────
# Strategy 1 (best on Colab): apt-get installs a system binary with all shared
#   libs already present — no wheel/conda dependency issues.
# Strategy 2: conda-forge build (larger download but self-contained).
# Strategy 3: openbabel-wheel pip package (last resort).
# ─────────────────────────────────────────────────────────────────────────────
def _obabel_ok():
    r = _run('obabel --version 2>&1')
    return 'Open Babel' in r.stdout or 'Open Babel' in r.stderr

print('\n📦 Installing OpenBabel (strategy 1/3 — apt-get) ...')
_run('apt-get update -qq')
_run('apt-get install -y -qq openbabel libopenbabel-dev')

if _obabel_ok():
    print(f'✅ OpenBabel ready via apt-get: {_run("obabel --version 2>&1 | head -1").stdout.strip()}')
else:
    print('⚠️  apt-get failed. Trying strategy 2/3 — conda-forge ...')
    if '/opt/miniforge/bin' not in os.environ.get('PATH', ''):
        _run(
            'wget -q https://github.com/conda-forge/miniforge/releases/latest/download/'
            'Miniforge3-Linux-x86_64.sh -O miniforge.sh && '
            'bash miniforge.sh -b -p /opt/miniforge'
        )
        os.environ['PATH'] = '/opt/miniforge/bin:' + os.environ['PATH']
    _run('conda install -y -c conda-forge openbabel 2>&1 | tail -5',
         env={**os.environ})
    if _obabel_ok():
        print('✅ OpenBabel ready via conda-forge.')
    else:
        print('⚠️  conda-forge failed. Trying strategy 3/3 — pip openbabel-wheel ...')
        _run('apt-get install -y -qq libopenbabel7 libopenbabel-dev libeigen3-dev')
        _run('pip install -q openbabel-wheel')
        if _obabel_ok():
            print('✅ OpenBabel ready via pip wheel.')
        else:
            raise RuntimeError(
                '❌ All three OpenBabel install strategies failed.\n'
                'Please restart the Colab runtime and try again, or '
                'manually run: !apt-get install -y openbabel'
            )

# ── 1d. PDB2PQR + PROPKA — protein-only pKa-correct protonation ──────────────
# PDB2PQR (the modern Python3 "pdb2pqr30" CLI) wraps PROPKA3 to predict the
# empirical pKa of every titratable residue (Asp/Glu/Lys/Arg/Cys/Tyr/His,
# N-/C-termini) from its actual structural microenvironment — H-bonding,
# desolvation, and charge-charge perturbation terms — then assigns the correct
# protonation state at TARGET_PH. This replaces OpenBabel for the PROTEIN-ONLY
# fragment because OpenBabel has no pKa model at all: it adds H's by valence/
# geometry heuristics only, and routinely gets His tautomer state and
# borderline Asp/Glu/Lys ionisation wrong near physiological pH.
#
# Cofactors are NOT touched by PDB2PQR: its force-field template libraries
# (AMBER, CHARMM, PARSE, TYL06, SWANSON) only contain standard amino-acid /
# nucleic-acid topologies. There is no FAD, NAD, heme, CoA, PLP, ATP, or SAM
# template, so PDB2PQR cannot process them. OpenBabel + prepare_ligand4.py
# remains the correct, unchanged path for organic cofactors.
print('\n📦 Installing pdb2pqr (bundles PROPKA3) via pip ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pdb2pqr'], check=True)

def _locate_pdb2pqr():
    for name in ('pdb2pqr30', 'pdb2pqr'):
        r = subprocess.run(['which', name], capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip():
            return r.stdout.strip()
    return None

PDB2PQR_BIN = _locate_pdb2pqr()
if not PDB2PQR_BIN:
    raise RuntimeError(
        '❌ pdb2pqr30 / pdb2pqr CLI not found on PATH after pip install.\n'
        '   Try: !pip install --force-reinstall pdb2pqr, then re-run this cell.'
    )

r_help = subprocess.run([PDB2PQR_BIN, '--help'], capture_output=True, text=True)
help_text = r_help.stdout + r_help.stderr
PDB2PQR_HAS_PROPKA = '--titration-state-method' in help_text
if not PDB2PQR_HAS_PROPKA:
    print(f'⚠️  {os.path.basename(PDB2PQR_BIN)} does not expose --titration-state-method — '
          f'check pdb2pqr version (PROPKA-driven titration may be unavailable).')

print(f'✅ pdb2pqr ready: {PDB2PQR_BIN}')
print(f'   PROPKA titration support: {"yes" if PDB2PQR_HAS_PROPKA else "NO — see warning above"}')

print('\n🎉 All dependencies installed successfully — MGLTools-free pipeline.')
print(f'   OpenBabel              : ready  (cofactor protonation)')
print(f'   pdb2pqr (+ PROPKA3)    : ready  (protein-only protonation)')
print(f'   prepare_receptor4.py   : {PREPARE_RECEPTOR4}')
print(f'   prepare_ligand4.py     : {PREPARE_LIGAND4}')
print(f'   prepare_flexreceptor4.py: {PREPARE_FLEXRECEPTOR4 or "not found (optional Step 3.5 unavailable)"}')




---
## 🔬 Step 1.5 — Set Target pH
**`TARGET_PH`** is a single unified parameter that governs protonation across the
entire pipeline. Set it **once here** — it is applied to **both** PDB fragments
produced by the split in further steps.

In [ ]:
#@title ⚙️ Set Target pH
#@markdown **Change `TARGET_PH` if your system requires a non-physiological pH.**
#@markdown Default: **7.4** (physiological). Common alternatives: 6.5 (tumour/lysosome),
#@markdown 7.0 (cytoplasm), 8.0 (mitochondrial matrix).

# ╔══════════════════════════════════════════════════════════════╗
# ║  USER-CONFIGURABLE: set your target pH here                  ║
# ║  Protein-only fragment → PDB2PQR + PROPKA3 (empirical pKa)   ║
# ║    pdb2pqr30 --with-ph TARGET_PH                              ║
# ║              --titration-state-method propka                ║
# ║    → prepare_receptor4.py -A checkhydrogens -C (preserve)    ║
# ║  Cofactor fragment(s)   → OpenBabel (no pKa model available  ║
# ║    for non-standard chemotypes)                              ║
# ║    obabel -p TARGET_PH                                       ║
# ║    → prepare_ligand4.py (no -A, Gasteiger charges)            ║
# ╚══════════════════════════════════════════════════════════════╝
TARGET_PH = 7.4  #@param {type:"number"}

if not (0.0 <= TARGET_PH <= 14.0):
    raise ValueError(f'TARGET_PH must be between 0 and 14 (got {TARGET_PH}).')

print(f'✅ TARGET_PH = {TARGET_PH}')
print(f'   → Protein-only protonation  : PDB2PQR + PROPKA3 (--with-ph {TARGET_PH})')
print(f'                                  empirical per-residue pKa, not geometry heuristics')
print(f'   → Protein charge handling   : prepare_receptor4.py -A checkhydrogens -C')
print(f'                                  (preserves PDB2PQR/AMBER charges, adds none new)')
print(f'   → Cofactor protonation      : OpenBabel (obabel -p {TARGET_PH})')
print(f'   → Cofactor charge handling  : prepare_ligand4.py (no -A) → Gasteiger')


---
## 📥 Step 2 — Load Your Protein Structure
Provide your receptor PDB file by uploading from your PC.

> ⚠️ **Important for "Around ligand" mode:** Ligand coordinates are captured here,
> *before* `prepare_receptor` strips HETATM records. Do not skip this step.
>
> ✅ **After uploading**, run the **"Select Non-Metallic Cofactors to Retain"** cell if you want
> to choose which cofactors (FAD, NAD, ZN, HEM, etc.) should be embedded in the
> receptor PDBQT and included in the grid box calculation.

In [ ]:
#@title ▶ Configure PDB Source
#@markdown ### Receptor name — the single name used for *every* output file:
#@markdown This one name drives the prepared receptor `.pdbqt`, the optional
#@markdown Step 3.5 rigid/flex split, the `.gpf`/`.glg` log, and every
#@markdown AutoGrid4 `.map` file.
receptor_name = "" #@param {type:"string"}

import os
work_dir = '/content/receptor_prep'
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

if not receptor_name.strip():
    raise ValueError('receptor_name cannot be blank — every output file in this pipeline is keyed off it.')

print(f'✅ Working directory : {work_dir}')
print(f'🧬 Receptor name     : {receptor_name}   (used for ALL outputs — .pdbqt, .gpf, .C.map, ...)')
print('➡️  Run the next cell to upload your PDB file.')


In [ ]:
#@title ▶ Upload PDB File
#@markdown Upload your receptor `.pdb` file from your PC.
#@markdown Ligand / cofactor HETATM coordinates are captured here
#@markdown for use in **"Around ligand"** grid box mode (Step 6).

import os, shutil

raw_pdb = os.path.join(work_dir, 'receptor_raw.pdb')

from google.colab import files
print('📂 Select your .pdb file from the file picker below.')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded. Please re-run this cell and select a .pdb file.')
for fname in uploaded.keys():
    shutil.move(fname, raw_pdb)
print(f'\n✅ File saved as: {raw_pdb}')

# ── Detect chains present in the file ────────────────────────────────────────
chain_info = {}
with open(raw_pdb) as fh:
    for line in fh:
        if line.startswith(('ATOM', 'HETATM')):
            try:
                chain   = line[21]
                resname = line[17:20].strip()
                resnum  = int(line[22:26].strip())
                if chain not in chain_info:
                    chain_info[chain] = {'residues': set(), 'types': set()}
                chain_info[chain]['residues'].add((resnum, resname))
                chain_info[chain]['types'].add('HETATM' if line.startswith('HETATM') else 'ATOM')
            except (ValueError, IndexError):
                pass

print(f'\n🔍 Chains detected in PDB file:')
print(f'  {"Chain":>6}  {"Residues":>8}  Types  [Sample residues]')
print('  ' + '-'*55)
for ch in sorted(chain_info.keys()):
    n_res    = len(chain_info[ch]['residues'])
    types    = ', '.join(sorted(chain_info[ch]['types']))
    resnames = sorted(set(rn for _, rn in chain_info[ch]['residues']))
    sample   = ', '.join(resnames[:6]) + (' ...' if len(resnames) > 6 else '')
    print(f'  {ch:>6}  {n_res:>8}  {types:5}  [{sample}]')

# ── Sanity check ──────────────────────────────────────────────────────────────
size_kb    = os.path.getsize(raw_pdb) / 1024
with open(raw_pdb) as fh:
    atom_count = sum(1 for l in fh if l.startswith(('ATOM', 'HETATM')))
print(f'\n📊 File size   : {size_kb:.1f} KB')
print(f'📊 ATOM/HETATM : {atom_count} records')

# ── Capture HETATM molecule coordinates (ligands / cofactors) ─────────────────
# These are captured NOW before prepare_receptor strips them.
# Required for "Around ligand" grid box mode in Step 6.
COMMON_METALS = {
    'ZN','FE','MG','CA','MN','CU','NI','CO','CD','HG',
    'PT','K','NA','CS','RB','AU','AG','BA','SR','LI','SE'
}
WATER_NAMES = {'HOH','WAT','H2O','DOD','TIP','SOL'}

ligand_coords_store = {}   # {resname: {'chain', 'resnum', 'coords': [(x,y,z)]}}

with open(raw_pdb) as fh:
    for line in fh:
        if not line.startswith('HETATM'):
            continue
        try:
            resname = line[17:20].strip()
            chain   = line[21]
            resnum  = int(line[22:26])
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        except (ValueError, IndexError):
            continue
        if resname in WATER_NAMES:
            continue
        if resname not in ligand_coords_store:
            ligand_coords_store[resname] = {
                'chain': chain, 'resnum': resnum, 'coords': []
            }
        ligand_coords_store[resname]['coords'].append((x, y, z))

if ligand_coords_store:
    print('\n💊 HETATM molecules detected (excluding water):')
    print(f'  {"Name":>6}  {"Chain":>5}  {"Atoms":>5}  Category')
    print('  ' + '-'*42)
    for rn, info in sorted(ligand_coords_store.items()):
        cat = 'Metal ion' if rn in COMMON_METALS else 'Ligand / Cofactor'
        print(f'  {rn:>6}  {info["chain"]:>5}  {len(info["coords"]):>5}  {cat}')
    print('\n  ℹ️  Use any of these residue names in "Around ligand" mode (Step 6).')
else:
    print('\n  ℹ️  No HETATM ligands or cofactors detected — "Around ligand" mode will not be available.')

In [ ]:
#@title ▶ Select Non-Metallic Cofactors to Retain in Receptor PDBQT
#@markdown ### 🧪 Cofactor Retention — Optional
#@markdown Enter residue names to **keep** in the final receptor PDBQT
#@markdown (comma-separated, e.g. `FAD, NAD` or `ZN, HEM`). Leave blank to skip.
#@markdown ---
#@markdown **Residue names to retain** *(3-letter codes)*:
cofactors_to_retain_input = "" #@param {type:"string"}
#@markdown > 💡 Common examples: FAD, FMN, NAD, NAP, COA, PLP, TPP, HEM, HEC,
#@markdown > ATP, ADP, SAM, BH4, PQQ, F42, NAG.

import os, re, subprocess, sys, shutil

# ── Guard: require Step 1 to have completed ───────────────────────────────────
if 'PREPARE_LIGAND4' not in dir() or not PREPARE_LIGAND4:
    raise RuntimeError(
        '❌ PREPARE_LIGAND4 not set — did Step 1 complete?\n'
        '   Re-run Step 1 to install ADFRsuite and locate prepare_ligand4.py.'
    )
if 'PREPARE_RECEPTOR4' not in dir() or not PREPARE_RECEPTOR4:
    raise RuntimeError(
        '❌ PREPARE_RECEPTOR4 not set — did Step 1 complete?\n'
        '   Re-run Step 1 to install ADFRsuite and locate prepare_receptor4.py.'
    )
if 'PDB2PQR_BIN' not in dir() or not PDB2PQR_BIN:
    raise RuntimeError(
        '❌ PDB2PQR_BIN not set — did Step 1 complete?\n'
        '   Re-run Step 1 to install pdb2pqr (PROPKA3-driven protein protonation).'
    )
if 'TARGET_PH' not in dir():
    TARGET_PH = 7.4
    print(f'ℹ️  TARGET_PH not set — defaulting to {TARGET_PH}. Run Step 1.5 to configure.')

# ── AD4-recognised elements ───────────────────────────────────────────────────
AD4_ELEMENTS = {
    'H', 'C', 'N', 'O', 'F', 'P', 'S',
    'CL', 'BR', 'I',
    'MG', 'CA', 'MN', 'FE', 'ZN', 'CU', 'CO', 'NI', 'SE', 'NA', 'K'
}
COMMON_METALS = {
    'ZN','FE','MG','CA','MN','CU','NI','CO','CD','HG',
    'PT','K','NA','CS','RB','AU','AG','BA','SR','LI','SE'
}

# ── Metal → AD4 type map ──────────────────────────────────────────────────────
METAL_AD4_TYPE = {
    'ZN':'Zn','FE':'Fe','MG':'Mg','CA':'Ca','MN':'Mn',
    'CU':'Cu','CO':'Co','NI':'Ni','SE':'Se','NA':'NA','K':'K',
}

# ── Shared helper: OpenBabel pH protonation of a PDB fragment ────────────────
def obabel_protonate(in_pdb, out_pdb, target_ph, label):
    """
    Run OpenBabel -p TARGET_PH on a PDB fragment (protein-only OR a cofactor
    fragment). Both fragments from the Step-2.5 split go through this SAME
    function so the entire receptor — protein and retained cofactors alike —
    is protonated at one consistent, user-defined pH before either ADFRsuite
    script ever sees it.

    -p <pH>  THE correct flag: calls OBPhModel to assign ionisation state
             at the given pH, removes existing H, adds new H accordingly.
    ⚠️  Do NOT use -d (strips H only) or -h (pH-unaware neutral addition).
    """
    h_before = 0
    with open(in_pdb) as fh:
        h_before = sum(1 for l in fh
                        if l.startswith(('ATOM', 'HETATM')) and l[12:16].strip().startswith('H'))

    subprocess.run(
        f'obabel {in_pdb} -O {out_pdb} -p {target_ph}',
        shell=True, capture_output=True, text=True
    )
    if not os.path.exists(out_pdb) or os.path.getsize(out_pdb) == 0:
        shutil.copy(in_pdb, out_pdb)
        print(f'  ⚠️  {label}: OpenBabel protonation failed — using unprotonated fragment.')
        return False

    with open(out_pdb) as fh:
        h_after = sum(1 for l in fh
                       if l.startswith(('ATOM', 'HETATM')) and l[12:16].strip().startswith('H'))
    print(f'  🔬 {label}: OpenBabel pH {target_ph} — '
          f'ΔH = {h_after - h_before:+d}  ({h_after} H in protonated fragment)')
    return True


# ── Protein-only pipeline: PDB2PQR + PROPKA3 (empirical pKa) ─────────────────
def pdb2pqr_protonate(in_pdb, out_pqr, target_ph, pdb2pqr_bin, work_dir, label='protein-only'):
    """
    Protonate the PROTEIN-ONLY fragment via PDB2PQR, using PROPKA3 to predict
    the empirical pKa of every titratable residue (Asp/Glu/Lys/Arg/Cys/Tyr/His,
    N-/C-termini) from the actual 3D structure (H-bonding, desolvation,
    charge-charge perturbation), then assigning the correct protonation state
    at `target_ph`. Unlike OpenBabel, this is NOT a geometry/valence heuristic —
    it is a real structure-based pKa calculation.

    PDB2PQR writes AMBER force-field partial charges directly into the
    resulting .pqr — these are carried through to `prepare_receptor4.py -C`
    untouched (no Gasteiger recalculation).

    --ff=AMBER                        : explicit-H, well-validated atomic charges
    --with-ph=<target_ph>             : protonate at the user-defined pH
    --titration-state-method=propka   : empirical pKa, not default rule-based states
    --keep-chain                      : preserve original chain IDs (needed for
                                         the legacy-format .pqr columns that
                                         prepare_receptor4.py's MolKit reader expects)
    --drop-water                      : remove crystallographic waters here rather
                                         than relying solely on prepare_receptor4.py's
                                         -U waters cleanup downstream

    Returns
    -------
    (out_pqr, net_charge)  on success
    Raises RuntimeError on failure — protonation correctness is load-bearing for
    -C charge preservation downstream, so this does NOT silently fall back.
    """
    cmd = [
        pdb2pqr_bin,
        '--ff=AMBER',
        f'--with-ph={target_ph}',
        '--titration-state-method=propka',
        '--keep-chain',
        '--drop-water',
        in_pdb, out_pqr,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=work_dir)

    if not os.path.exists(out_pqr) or os.path.getsize(out_pqr) == 0:
        print(f'  ❌ {label}: PDB2PQR failed.')
        if result.stdout.strip():
            print(f'     STDOUT: {result.stdout[-800:]}')
        if result.stderr.strip():
            print(f'     STDERR: {result.stderr[-800:]}')
        raise RuntimeError(
            f'PDB2PQR protonation of {label} failed — see output above. '
            f'Common causes: missing/duplicate atoms, multi-model PDB, or a '
            f'pdb2pqr/PROPKA version mismatch. Re-check Step 1 install.'
        )

    # ── Robust free-format PQR parsing (NOT fixed-column) ────────────────────
    # PDB2PQR's .pqr uses whitespace-delimited fields, NOT strict PDB columns:
    #   recordName serial atomName resName [chainID] resSeq x y z charge radius
    # Charge is always the second-to-last token, radius the last — this holds
    # regardless of how many digits the charge/coordinate fields use, so it's
    # safe even when fixed-column parsers would misalign.
    net_charge = 0.0
    n_atoms    = 0
    with open(out_pqr) as fh:
        for line in fh:
            if not line.startswith(('ATOM', 'HETATM')):
                continue
            fields = line.split()
            try:
                net_charge += float(fields[-2])
                n_atoms += 1
            except (ValueError, IndexError):
                continue

    print(f'  🔬 {label}: PDB2PQR + PROPKA3 (AMBER, pH {target_ph}) — '
          f'{n_atoms} atoms, net charge = {net_charge:+.3f} e')

    # ── Best-effort PROPKA pKa summary (bonus diagnostic, never blocking) ────
    propka_path = os.path.splitext(out_pqr)[0] + '.propka'
    if os.path.exists(propka_path):
        try:
            import re as _re
            shifted = []
            row_re = _re.compile(
                r'^\s*(ASP|GLU|LYS|ARG|HIS|CYS|TYR|N\+|C-)\s+(\d+)\s+(\w)\s+'
                r'([\d.]+)\s+([\d.]+)'
            )
            with open(propka_path) as fh:
                for line in fh:
                    m = row_re.match(line)
                    if not m:
                        continue
                    resn, resi, chain, pka, model_pka = m.groups()
                    pka, model_pka = float(pka), float(model_pka)
                    if abs(pka - model_pka) > 1.0:
                        shifted.append((resn, resi, chain, pka))
            if shifted:
                print(f'     ⚡ PROPKA: {len(shifted)} residue(s) with pKa shifted >1.0 '
                      f'unit from model value (microenvironment-perturbed):')
                for resn, resi, chain, pka in shifted[:6]:
                    print(f'        {resn}{resi}{chain}: predicted pKa {pka:.2f}')
                if len(shifted) > 6:
                    print(f'        ... and {len(shifted) - 6} more (see {os.path.basename(propka_path)})')
        except Exception:
            pass  # diagnostic only — never let summary parsing break the pipeline

    return out_pqr, net_charge


# ── Organic cofactor pipeline: OpenBabel pH → ADFRsuite prepare_ligand4.py ────
def prepare_cofactor_pdbqt(rn, raw_lines_rn, work_dir, prepare_ligand4,
                            target_ph, serial_start):
    """
    Convert a raw HETATM PDB fragment → AD4-typed, Gasteiger-charged PDBQT.

    Pipeline
    --------
      _cof_{rn}.pdb   (original crystal coords, no H)
      → OpenBabel -p target_ph    (pH-aware ionisation + H addition)
      → _cof_{rn}_prot.pdb
      → prepare_ligand4.py -Z -U nphs_lps   (ADFRsuite, run directly, NO -A flag)
          (no -A flag : prepare_ligand4.py's -A only supports
                         bonds_hydrogens / bonds / hydrogens — there is no
                         'checkhydrogens' repair type for ligand prep (that
                         belongs to AD4ReceptorPreparation, not
                         AD4LigandPreparation). Default repairs="" means
                         "make no repairs" — it trusts the OpenBabel-protonated
                         structure as-is and types OA/NA/HD/SA/A/C directly
                         from the H's already present.)
          (-Z         : rigid — no ROOT/BRANCH/TORSDOF records)
          (-U nphs_lps: merges non-polar H onto C for Gasteiger)
      → _cof_{rn}.pdbqt
      → HETATM lines with original rn/chain/resnum patched back

    Returns
    -------
    (list_of_pdbqt_lines, serial_end)  on success
    (None, serial_start)               on failure (caller handles fallback)
    """
    frag_pdb   = os.path.join(work_dir, f'_cof_{rn}.pdb')
    frag_prot  = os.path.join(work_dir, f'_cof_{rn}_prot.pdb')
    frag_pdbqt = os.path.join(work_dir, f'_cof_{rn}.pdbqt')

    # Original metadata from first raw line (chain / resnum / icode)
    orig_line   = raw_lines_rn[0]
    orig_chain  = orig_line[21]    if len(orig_line) > 21 else 'A'
    orig_resnum = orig_line[22:26] if len(orig_line) > 25 else '   1'
    orig_icode  = orig_line[26]    if len(orig_line) > 26 else ' '

    # ── Step A: write fragment PDB ────────────────────────────────────────────
    with open(frag_pdb, 'w') as fh:
        for line in raw_lines_rn:
            fh.write(line)
        fh.write('END\n')

    # ── Step B: OpenBabel pH protonation ─────────────────────────────────────
    obabel_protonate(frag_pdb, frag_prot, target_ph, rn)

    # ── Step C: prepare_ligand4.py (ADFRsuite, run directly) ──────────────────
    cmd_adt = [
        prepare_ligand4,
        '-l', frag_prot,
        '-o', frag_pdbqt,
        # NO -A flag: prepare_ligand4.py's -A only accepts bonds_hydrogens/bonds/
        # hydrogens (per its own usage()) — there is no 'checkhydrogens' option
        # for ligand prep. Omitting -A defaults to repairs="" ("do no repairs"),
        # which is exactly right here: OpenBabel already protonated frag_prot,
        # so we want AD4 typing read straight off the existing H's, untouched.
        '-Z',                   # rigid: no ROOT/BRANCH/TORSDOF records
        '-U', 'nphs_lps',       # merge non-polar H onto C (Gasteiger standard)
    ]
    r_adt = subprocess.run(cmd_adt, capture_output=True, text=True)

    if not os.path.exists(frag_pdbqt) or os.path.getsize(frag_pdbqt) == 0:
        print(f'  ❌ {rn}: prepare_ligand4.py failed.')
        if r_adt.stdout.strip():
            print(f'     STDOUT: {r_adt.stdout[-400:]}')
        if r_adt.stderr.strip():
            print(f'     STDERR: {r_adt.stderr[-400:]}')
        return None, serial_start

    # ── Step D: parse PDBQT — keep only ATOM/HETATM records ──────────────────
    with open(frag_pdbqt) as fh:
        raw_pdbqt = fh.readlines()
    atom_records = [l for l in raw_pdbqt if l.startswith(('ATOM', 'HETATM'))]

    if not atom_records:
        print(f'  ❌ {rn}: No ATOM/HETATM records in prepare_ligand4.py output.')
        return None, serial_start

    # ── Step E: patch residue name / chain / resnum back to original ─────────
    # prepare_ligand4.py may rename the residue (e.g. LIG, UNL) and renumber.
    # Restore rn / chain / resnum from original crystal structure records.
    # Coordinates and AD4 type/charge fields come from prepare_ligand4.py.
    patched_lines = []
    serial = serial_start
    for pl in atom_records:
        if len(pl) < 77:
            continue
        serial += 1

        # Fields from prepare_ligand4.py PDBQT output
        atom_name  = pl[12:16]           # e.g. ' C4A'
        alt_loc    = pl[16]              # usually ' '
        x_f        = pl[30:38]           # coordinates (from protonated PDB
        y_f        = pl[38:46]           #   → heavy atoms = crystal coords,
        z_f        = pl[46:54]           #   → H atoms = OB-computed)
        occ_f      = pl[54:60] if len(pl) > 59 else '  1.00'
        bfac_f     = pl[60:66] if len(pl) > 65 else '  0.00'
        charge_str = pl[70:76] if len(pl) > 75 else '+0.000'
        ad4type    = pl[77:79].strip()   if len(pl) > 77 else 'C'

        new_line = (
            f'HETATM{serial:5d} {atom_name}{alt_loc}'
            f'{rn:<3} {orig_chain}{orig_resnum}{orig_icode}   '
            f'{x_f}{y_f}{z_f}'
            f'{occ_f}{bfac_f}    '
            f'{charge_str} {ad4type:<2}\n'
        )
        patched_lines.append(new_line)

    # ── Report ────────────────────────────────────────────────────────────────
    ad4_types_found = set()
    charges_found   = []
    for pl in atom_records:
        if len(pl) >= 79:
            ad4_types_found.add(pl[77:79].strip())
        try:
            charges_found.append(float(pl[70:76].strip()))
        except (ValueError, IndexError):
            pass
    net_q = sum(charges_found)

    # Key AD4 types that should be present for H-bond active cofactors
    expected_hb_types = {'OA', 'HD'}
    missing_hb = expected_hb_types - ad4_types_found
    hb_note = '' if not missing_hb else f'  ⚠️ expected H-bond types {missing_hb} not found'

    print(f'  ✅ {rn}: {len(patched_lines)} atoms  '
          f'AD4={sorted(ad4_types_found)}  '
          f'net={net_q:+.3f} e{hb_note}')

    return patched_lines, serial


# ── Universal protein-only protonation (PDB2PQR + PROPKA3) ───────────────────
# Runs exactly ONCE, regardless of whether any cofactor is retained, because
# Step 3's prepare_receptor4.py ALWAYS needs a PROPKA-protonated, AMBER-charged
# protein-only .pqr. The protein-only fragment here is ATOM records ONLY —
# every HETATM (waters, unselected metals, ions, and any cofactor regardless of
# whether the user retains it) is stripped before PDB2PQR ever sees the file.
# PDB2PQR's force-field templates (AMBER/CHARMM/PARSE/TYL06/SWANSON) only cover
# standard amino acids — feeding it HETATM cofactors would either error out or
# silently drop them, so cofactors are handled entirely separately below via
# OpenBabel + prepare_ligand4.py, exactly as before.
protein_only_pdb = os.path.join(work_dir, 'receptor_protein_only.pdb')
with open(raw_pdb) as fi, open(protein_only_pdb, 'w') as fo:
    for line in fi:
        if line.startswith('HETATM'):
            continue
        fo.write(line)

print(f'🔬 Protonating protein-only PDB at pH {TARGET_PH} (PDB2PQR + PROPKA3) ...')
protein_only_pqr = os.path.join(work_dir, 'receptor_protein_only.pqr')
_, protein_net_charge = pdb2pqr_protonate(
    protein_only_pdb, protein_only_pqr, TARGET_PH, PDB2PQR_BIN, work_dir
)
pdb_for_receptor = protein_only_pqr
print()

# ── Guard: require Upload PDB cell to have run ────────────────────────────────
if 'ligand_coords_store' not in dir() or not ligand_coords_store:
    print('⚠️  No HETATM molecules detected (or Upload PDB cell not yet run).')
    cofactors_to_retain    = set()
    cofactors_cleanup_flag = 'nphs_lps_waters_nonstdres'
    cofactor_pdbqt_lines   = []
else:
    print('💊 HETATM residues available in your PDB:')
    print(f'  {"Name":>6}  {"Atoms":>5}  {"Chain":>5}  Category')
    print('  ' + '─'*46)
    for rn, info in sorted(ligand_coords_store.items()):
        cat = '🔩 Metal ion' if rn in COMMON_METALS else '🧪 Cofactor / Ligand'
        print(f'  {rn:>6}  {len(info["coords"]):>5}  {info["chain"]:>5}  {cat}')
    print()

    # ── Parse user input ──────────────────────────────────────────────────────
    cofactors_to_retain = set()
    if cofactors_to_retain_input.strip():
        for tok in re.split(r'[,\s]+', cofactors_to_retain_input.strip().upper()):
            tok = tok.strip()
            if tok:
                cofactors_to_retain.add(tok)

    if not cofactors_to_retain:
        print('ℹ️  No cofactors selected — all HETATM stripped by prepare_receptor4.py.')
        cofactors_to_retain    = set()
        cofactors_cleanup_flag = 'nphs_lps_waters_nonstdres'
        cofactor_pdbqt_lines   = []
        # pdb_for_receptor already set above (protein_only_pqr) — nothing more to do.
    else:
        available = set(ligand_coords_store.keys())
        valid     = cofactors_to_retain & available
        not_found = cofactors_to_retain - available
        if not_found:
            print(f'⚠️  Not found in PDB (ignored): {sorted(not_found)}')
        if not valid:
            print('❌ None of the selected residues were found.')
            cofactors_to_retain    = set()
            cofactors_cleanup_flag = 'nphs_lps_waters_nonstdres'
            cofactor_pdbqt_lines   = []
            # pdb_for_receptor already set above (protein_only_pqr) — nothing more to do.
        else:
            # ── Element compatibility check ────────────────────────────────
            print('🔬 Checking AD4 element compatibility ...')
            unsupported = {}
            with open(raw_pdb) as fh:
                for line in fh:
                    if not line.startswith('HETATM'):
                        continue
                    rn = line[17:20].strip().upper()
                    if rn not in valid:
                        continue
                    el = line[76:78].strip().upper() if len(line) > 76 else ''
                    if not el:
                        el = re.sub(r'[^A-Za-z]', '', line[12:16].strip())[:2].upper()
                    if el and el not in AD4_ELEMENTS:
                        unsupported.setdefault(rn, set()).add(el)
            if unsupported:
                print('  ⚠️  Unsupported elements (atoms skipped):')
                for rn, els in sorted(unsupported.items()):
                    print(f'     {rn}: {sorted(els)}')

            # ── Split: metals vs organic cofactors ────────────────────────
            metal_cofactors   = valid & COMMON_METALS
            organic_cofactors = valid - COMMON_METALS
            print(f'\n   Metal cofactors    (charge=0.000)            : {sorted(metal_cofactors) or "none"}')
            print(f'   Organic cofactors  (ADFRsuite prepare_ligand4.py) : {sorted(organic_cofactors) or "none"}')
            print(f'   Target pH — protein (PDB2PQR/PROPKA3) & cofactors (OpenBabel) : {TARGET_PH}')
            # NOTE: protein-only protonation already done ONCE, above, for ALL
            # branches (pdb_for_receptor = protein_only_pqr). Rationale for why
            # it must be ATOM-only (no HETATM at all, not just non-retained ones):
            # prepare_receptor4.py applies an internal residue-name filter (MolKit
            # vocabulary) that silently discards any residue name outside its
            # standard amino-acid dictionary regardless of -U settings, AND
            # PDB2PQR's AMBER/CHARMM/PARSE templates have no cofactor parameters
            # at all — so retained cofactors are prepared entirely separately
            # below (OpenBabel + prepare_ligand4.py) and re-appended post-hoc.

            # ── Prepare PDBQT records for each retained cofactor ──────────────
            cofactor_pdbqt_lines = []
            serial_offset        = 99000

            print(f'\n🔧 Preparing cofactor PDBQT records ...')
            for rn in sorted(valid):
                is_metal = rn in COMMON_METALS

                # Collect raw HETATM lines for this residue from the original PDB
                raw_lines_rn = [l for l in open(raw_pdb)
                                if l.startswith('HETATM')
                                and l[17:20].strip().upper() == rn]
                if not raw_lines_rn:
                    continue

                if is_metal:
                    # ── Metal ions: keep crystal coordinates, charge=0.000 ────
                    for line in raw_lines_rn:
                        el = line[76:78].strip().upper() if len(line) > 76 else ''
                        if not el:
                            el = re.sub(r'[^A-Za-z]', '', line[12:16].strip())[:2].upper()
                        if el not in AD4_ELEMENTS:
                            continue
                        ad4type = METAL_AD4_TYPE.get(el, el.capitalize())
                        a_f    = line[12:16]
                        alt    = line[16]   if len(line) > 16 else ' '
                        rn_f   = line[17:20]
                        ch     = line[21]   if len(line) > 21 else 'A'
                        seq_f  = line[22:26]
                        ic     = line[26]   if len(line) > 26 else ' '
                        x_f    = line[30:38]
                        y_f    = line[38:46]
                        z_f    = line[46:54]
                        occ    = line[54:60] if len(line) > 59 else '  1.00'
                        bfac   = line[60:66] if len(line) > 65 else '  0.00'
                        serial_offset += 1
                        pqt = (f'HETATM{serial_offset:5d} {a_f}{alt}'
                               f'{rn_f} {ch}{seq_f}{ic}   '
                               f'{x_f}{y_f}{z_f}'
                               f'{occ}{bfac}    '
                               f'+0.000 {ad4type:<2}\n')
                        cofactor_pdbqt_lines.append(pqt)
                    print(f'  🔩 {rn}: {len(raw_lines_rn)} atoms  '
                          f'charge=0.000  [{METAL_AD4_TYPE.get(rn, rn)}]')

                else:
                    # ── Organic cofactors: full prepare_ligand4.py pipeline ───
                    new_lines, serial_offset = prepare_cofactor_pdbqt(
                        rn, raw_lines_rn, work_dir,
                        PREPARE_LIGAND4,
                        TARGET_PH, serial_offset
                    )
                    if new_lines:
                        cofactor_pdbqt_lines.extend(new_lines)
                    else:
                        print(f'  ❌ {rn}: prepare_ligand4.py pipeline failed — cofactor NOT added.')
                        print(f'     Check that Step 1 completed (ADFRsuite installed) and re-run.')

            # ── Store pipeline variables for Step 3 ───────────────────────────
            # pdb_for_receptor already set above (protein_only_pqr) — unchanged here.
            cofactors_to_retain    = valid
            cofactors_cleanup_flag = 'nphs_lps_waters_nonstdres'

            # ── Summary ───────────────────────────────────────────────────────
            print(f'\n✅ Cofactor retention configured (two-pass, dual-tool protonation):')
            print(f'   Retained cofactors                     : {sorted(valid)}')
            print(f'   Target pH — protein (PDB2PQR/PROPKA3)  : {TARGET_PH}')
            print(f'   Target pH — cofactors (OpenBabel)      : {TARGET_PH}')
            for rn in sorted(valid):
                n_heavy = len(ligand_coords_store[rn]['coords'])
                lines_for_rn = [l for l in cofactor_pdbqt_lines
                                if l[17:20].strip() == rn]
                n_pdbqt = len(lines_for_rn)
                if rn in COMMON_METALS:
                    label = f'🔩 Metal   charge=0.000  [{METAL_AD4_TYPE.get(rn, rn)}]'
                elif n_pdbqt > 0:
                    ad4t = sorted(set(l[77:79].strip() for l in lines_for_rn if len(l) > 79))
                    label = (f'🧪 Organic  prepare_ligand4.py  '
                             f'{n_pdbqt} PDBQT atoms (incl. polar H)  '
                             f'AD4={ad4t}')
                else:
                    label = '❌ Organic  pipeline FAILED — not added'
                print(f'   └─ {rn:>6}  {n_heavy:>4} heavy atoms  {label}')
            print(f'\n   Protein-only PQR (PDB2PQR/PROPKA3)     : {os.path.basename(protein_only_pqr)}  '
                  f'(net charge {protein_net_charge:+.3f} e)')
            print(f'   Cofactor PDBQT records                  : {len(cofactor_pdbqt_lines)}')
            print(f'   (heavy atoms + polar H from pH {TARGET_PH} OpenBabel protonation)')
            print(f'\n⚙️  prepare_receptor4.py → protein only, -A checkhydrogens -C (preserve PQR charges).')
            print(f'   Cofactors appended post-prep: AD4 types + Gasteiger charges')
            print(f'   via prepare_ligand4.py (no -A flag — trusts OB protonation; not heuristic')
            print(f'   element-to-type mapping).')
            print(f'\n➡️  Proceed to Step 3 — Prepare Receptor')


---
## 🧪 Step 3 — Prepare Receptor


In [ ]:
#@title ▶ Prepare Receptor → PDBQT
#@markdown Prepares the `.pdbqt` file of the receptor.

import os, subprocess

# ── Shared helper: enforce correct TER/END placement + continuous serials ───
# Used here (Step 3) AND re-used in Step 3.5 (flexible-residue split), since
# prepare_flexreceptor4.py's rigid-part output has the same TER/END gaps.
def finalize_pdbqt_chain(pdbqt_path):
    """
    Post-process a receptor PDBQT so it round-trips cleanly through
    PDB-aware downstream tools (PyMOL, MDAnalysis, GROMACS pdb2gmx, etc.),
    which rely on TER/END to detect chain and residue boundaries:

      1. Renumber ALL ATOM/HETATM serials sequentially from 1. This removes
         the cofactor-prep stage's temporary 99000+ placeholder serials
         (used only to avoid collisions while cofactor fragments were
         prepared independently of the protein) and produces one
         continuous numbering scheme — the same convention
         prepare_flexreceptor4.py's own rigid-part output uses.
      2. Insert exactly one TER record at every chain-segment boundary:
           - between consecutive protein chains (ATOM chain ID changes)
           - between the last protein ATOM and the first HETATM
           - between each distinct HETATM residue (chain+resSeq+icode
             changes) — so every retained cofactor gets its own TER, not
             just the very last one
           - after the very last ATOM/HETATM record in the file
      3. Ensure the file ends with exactly one END record.

    Any pre-existing TER/END/ENDMDL/blank lines are discarded and rebuilt
    from scratch, so this is idempotent — safe to call on a file that
    already has correct TER/END placement (e.g. a protein-only receptor
    with no retained cofactors).

    Coordinates, atom names, AD4 types, and partial charges are untouched —
    only columns 7-11 (serial) are rewritten, and only on ATOM/HETATM lines.
    """
    with open(pdbqt_path) as fh:
        lines = fh.readlines()

    atom_lines = [l for l in lines if l.startswith(('ATOM', 'HETATM'))]
    if not atom_lines:
        return False

    def res_key(l):
        return l[21:27]   # chainID + resSeq + iCode, as one comparable token

    out    = []
    serial = 0
    prev   = None   # (is_atom, chainID, res_key) of the previously written record

    for line in atom_lines:
        is_atom = line.startswith('ATOM')
        chain   = line[21:22]
        key     = res_key(line)

        if prev is not None:
            p_is_atom, p_chain, p_key = prev
            boundary = (
                (p_is_atom and not is_atom) or                   # protein -> cofactor
                (p_is_atom and is_atom and chain != p_chain) or   # next protein chain
                (not p_is_atom and not is_atom and key != p_key)  # next cofactor residue
            )
            if boundary:
                prev_line = out[-1]
                serial += 1
                out.append(f'TER   {serial:5d}      {prev_line[17:20]} {prev_line[21:27]}\n')

        serial += 1
        out.append(f'{line[:6]}{serial:5d}{line[11:]}')
        prev = (is_atom, chain, key)

    # Final TER after the last record, then a single END.
    prev_line = out[-1]
    serial += 1
    out.append(f'TER   {serial:5d}      {prev_line[17:20]} {prev_line[21:27]}\n')
    out.append('END\n')

    with open(pdbqt_path, 'w') as fh:
        fh.writelines(out)
    return True


receptor_pdbqt = os.path.join(work_dir, f'{receptor_name}.pdbqt')

# Defaults if cofactor selection cell was skipped
pdb_for_receptor       = pdb_for_receptor       if 'pdb_for_receptor'       in dir() else raw_pdb
cofactors_cleanup_flag = cofactors_cleanup_flag  if 'cofactors_cleanup_flag' in dir() else 'nphs_lps_waters_nonstdres'
cofactor_pdbqt_lines   = cofactor_pdbqt_lines   if 'cofactor_pdbqt_lines'   in dir() else []

cmd = [
    PREPARE_RECEPTOR4,
    '-r', pdb_for_receptor,
    '-o', receptor_pdbqt,
    '-A', 'checkhydrogens',  # safety net only — PDB2PQR already added all H's
    '-C',                    # preserve PDB2PQR/PROPKA3/AMBER charges verbatim;
                             # add NO new (Gasteiger) charges to the protein
    '-U', cofactors_cleanup_flag,
    '-e'
]

print('🔧 Running prepare_receptor4.py (ADFRsuite, direct executable) ...')
print(f'   Input           : {os.path.basename(pdb_for_receptor)}  (PDB2PQR/PROPKA3-protonated, AMBER charges)')
print(f'   Charge handling : -C  (preserve verbatim — no Gasteiger recalculation)')
print(f'   Cleanup flags   : -{cofactors_cleanup_flag}')
result = subprocess.run(cmd, capture_output=True, text=True, cwd=work_dir)

if result.returncode != 0 or not os.path.exists(receptor_pdbqt):
    print('❌ prepare_receptor4.py failed:')
    print(result.stdout); print(result.stderr)
else:
    # ── Charge-conservation QC: independently verify -C actually preserved
    #    PDB2PQR's charges, since prepare_receptor4.py's own usage() only
    #    claims "possibly pqr" support and its MolKit reader predates PDB2PQR's
    #    modern free-format PQR convention. ─────────────────────────────────
    if 'protein_net_charge' in dir():
        pqr_net_charge = protein_net_charge
    else:
        # Re-derive independently (free-format split, NOT fixed columns) in
        # case this cell is re-run without re-running Step 2.5.
        pqr_net_charge = 0.0
        with open(pdb_for_receptor) as fh:
            for line in fh:
                if not line.startswith(('ATOM', 'HETATM')):
                    continue
                fields = line.split()
                try:
                    pqr_net_charge += float(fields[-2])
                except (ValueError, IndexError):
                    continue

    receptor_net_charge = 0.0
    with open(receptor_pdbqt) as fh:
        for line in fh:
            if not line.startswith('ATOM'):
                continue
            try:
                receptor_net_charge += float(line[70:76].strip())
            except (ValueError, IndexError):
                continue

    charge_delta = receptor_net_charge - pqr_net_charge
    print(f'\n🔍 Charge-conservation check (PQR → PDBQT, protein only):')
    print(f'   PQR net charge (PDB2PQR/PROPKA3, AMBER)  : {pqr_net_charge:+.3f} e')
    print(f'   PDBQT net charge (post -C, post nphs_lps merge) : {receptor_net_charge:+.3f} e')
    if abs(charge_delta) > 0.5:
        print(f'   ❌ Δ = {charge_delta:+.3f} e — LARGE mismatch! This suggests -C did NOT '
              f'preserve PQR charges (possible PQR column-parsing failure in MolKit\'s '
              f'reader, or Gasteiger was silently applied). Inspect {os.path.basename(receptor_pdbqt)} '
              f'manually before proceeding to docking.')
    elif abs(charge_delta) > 0.05:
        print(f'   ⚠️  Δ = {charge_delta:+.3f} e — small deviation, plausibly from -U cleanup '
              f'removing a few non-standard atoms/waters that carried residual charge. Worth a spot-check.')
    else:
        print(f'   ✅ Δ = {charge_delta:+.3f} e — charges conserved through nphs_lps merge as expected.')

    # ── Append cofactor HETATM records ────────────────────────────────────────
    if cofactor_pdbqt_lines:
        with open(receptor_pdbqt) as fh:
            content = fh.readlines()

        # Strip trailing TER / END / ENDMDL / blank lines written by
        # prepare_receptor4.py -- finalize_pdbqt_chain() (below) rebuilds the
        # correct TER/END placement from scratch once everything is appended,
        # so there's no need to preserve them here.
        while content and (
            content[-1].strip() in ('END', 'ENDMDL', '')
            or content[-1].lstrip().startswith('TER')
        ):
            content.pop()

        content.extend(cofactor_pdbqt_lines)

        with open(receptor_pdbqt, 'w') as fh:
            fh.writelines(content)

        # -- Renumber serials + insert TER after EVERY residue/chain boundary
        #    (protein<->cofactor AND between each individual cofactor), and
        #    ensure a single trailing END. This also replaces the cofactor
        #    block's temporary 99000+ placeholder serials (assigned in the
        #    cofactor-retention cell purely to avoid collisions while each
        #    cofactor fragment was prepared independently of the protein)
        #    with serials that continue straight on from the protein's final
        #    atom number -- the same convention prepare_flexreceptor4.py uses
        #    for its own rigid-part output. ----------------------------------
        finalize_pdbqt_chain(receptor_pdbqt)
        print(f'\U0001f522 Serials renumbered continuously from 1; TER inserted after '
              f'the protein and after each retained cofactor residue; single '
              f'trailing END written.')

        # ── Verification: confirm all cofactor residues present ───────────────
        found_het_res = set()
        with open(receptor_pdbqt) as fh:
            for line in fh:
                if line.startswith('HETATM'):
                    found_het_res.add(line[17:20].strip())
        expected = set(l[17:20].strip() for l in cofactor_pdbqt_lines)
        missing  = expected - found_het_res
        if missing:
            print(f'❌ Cofactor residues missing from PDBQT: {sorted(missing)}')
        else:
            print(f'✅ All cofactor residues confirmed in PDBQT: {sorted(expected)}')
        print(f'   Appended {len(cofactor_pdbqt_lines)} cofactor HETATM records.')

        # ── Charge sanity report ──────────────────────────────────────────────
        print('\n📊 Charge summary for appended cofactors:')
        res_q = {}
        for line in cofactor_pdbqt_lines:
            rn = line[17:20].strip()
            try:
                q = float(line[70:76].strip())
            except (ValueError, IndexError):
                q = 0.0
            res_q.setdefault(rn, []).append(q)
        for rn, qs in sorted(res_q.items()):
            nonzero = sum(1 for q in qs if abs(q) > 0.001)
            net     = sum(qs)
            charge_src = 'charge=0.000' if nonzero == 0 else 'Gasteiger'
            print(f'   {rn:>6}  {len(qs):>4} atoms  '
                  f'{nonzero:>4} non-zero  net={net:+.3f} e  [{charge_src}]')
    else:
        # Still normalize TER/END for consistency, even with zero cofactors
        # (prepare_receptor4.py is usually already correct here, but this
        # makes the guarantee unconditional rather than tool-version-dependent).
        finalize_pdbqt_chain(receptor_pdbqt)
        print('ℹ️  No cofactors to append — standard protein-only receptor.')

    size_kb = os.path.getsize(receptor_pdbqt) / 1024
    with open(receptor_pdbqt) as fh:
        lines = fh.readlines()
    n_atom   = sum(1 for l in lines if l.startswith('ATOM'))
    n_hetatm = sum(1 for l in lines if l.startswith('HETATM'))

    print(f'\n✅ {receptor_name}.pdbqt  →  {size_kb:.1f} KB  (single canonical receptor file)')
    print(f'   ATOM records    : {n_atom}  (protein, PROPKA3-protonated, AMBER charges)')
    print(f'   HETATM records  : {n_hetatm}  (retained cofactors, Gasteiger charges)')
    print(f'\nℹ️  Ligand coordinates from Step 2 remain available for "Around ligand" grid mode.')
    print(f'ℹ️  {receptor_name}.pdbqt is the canonical receptor for the optional Step 3.5')
    print(f'   (flexible residues) and for Steps 4–7 — regardless of whether Step 3.5 is used.')



---
## 〰 Step 3.5 — Define Flexible Receptor Residues (Optional)


> ⏭️ **Fully optional.** Leave both selection fields blank in the next cell
> to skip this step entirely and proceed with a standard rigid-only receptor.

> ⚠️ **Input note:** The input is given the **full merged** PDBQT (protein +
> cofactor HETATM records), not an ATOM-only fragment.
---


In [ ]:
#@title ▶ Inspect Residues Available for Flexibility
#@markdown Lists every residue in the **Step 3 merged receptor**,
#@markdown with the exact selection token needed in the next cell.

import os

if 'receptor_pdbqt' not in dir() or not os.path.exists(receptor_pdbqt):
    raise RuntimeError('❌ receptor_pdbqt not found — run Step 3 (Prepare Receptor) first.')

mol_name = os.path.splitext(os.path.basename(receptor_pdbqt))[0]   # == receptor_name

seen = {}
with open(receptor_pdbqt) as fh:
    for line in fh:
        if not line.startswith(('ATOM', 'HETATM')):
            continue
        resname = line[17:20].strip()
        chain   = line[21].strip() or '_'
        resnum  = line[22:26].strip()
        inscode = line[26].strip()
        rectype = 'HETATM' if line.startswith('HETATM') else 'ATOM'
        key = (chain, resname, resnum, inscode)
        if key not in seen:
            seen[key] = rectype

chains = sorted({k[0] for k in seen})

print(f'Receptor   : {os.path.basename(receptor_pdbqt)}')
print(f'Mol name   : {mol_name}   (selection-string prefix — matches receptor_name)')
print(f'Chains     : {chains}\n')
print(f'{"Chain":<6}  {"ResName":<8}  {"ResNum":<6}  {"Type":<7}  {"→ Token":<16}  Full selection string')
print('-' * 85)
for (chain, resname, resnum, inscode), rectype in sorted(seen.items()):
    token = f'{resname}{resnum}{inscode}'
    full  = f'{mol_name}:{chain}:{token}'
    print(f'{chain:<6}  {resname:<8}  {resnum:<6}  {rectype:<7}  {token:<16}  {full}')

print('\n' + '=' * 85)
if len(chains) > 1:
    print(f'⚠️  Multi-chain receptor detected ({len(chains)} chains) — group tokens by chain in Method A below.')
print('Copy tokens / selection strings into the next cell.')
print('Leave BOTH fields blank in the next cell to skip flexible-residue prep entirely.')


In [ ]:
#@title ▶ Define Flexible Residues (leave both blank to skip)
#@markdown ---
#@markdown ### 🅐 Method A — Same-chain groups
#@markdown Format: `{receptor_name}:Chain:RES#_RES#`  — comma-separate multiple chains,
#@markdown e.g. `1u72:A:VAL3_THR8` or `1u72:B:VAL2_HIS5`
#@markdown Leave blank if using Method B, or leave BOTH blank to skip flexible residues entirely.
FLEX_METHOD_A = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🅑 Method B — Individual residues across chains
#@markdown Format: `{receptor_name}:Chain:RES#,{receptor_name}:Chain:RES#`  — one residue per entry.
#@markdown Leave blank if using Method A, or leave BOTH blank to skip flexible residues entirely.
FLEX_METHOD_B = "" #@param {type:"string"}

import os, subprocess, shutil

flex_method_a = FLEX_METHOD_A.strip()
flex_method_b = FLEX_METHOD_B.strip()

flexible_residues_defined = False
flex_pdbqt = None   # only set if flexible prep succeeds

if 'receptor_pdbqt' not in dir() or not os.path.exists(receptor_pdbqt):
    raise RuntimeError('❌ receptor_pdbqt not found — run Step 3 (Prepare Receptor) first.')

# ── Case 1: ambiguous input — both methods filled ─────────────────────────────
if flex_method_a and flex_method_b:
    print('⚠️  Both Method A and Method B are filled in — ambiguous input.')
    print('    Clear one of them and re-run this cell if you want flexible residues.')
    print(f'\nℹ️  Skipping flexible-residue prep. {receptor_name}.pdbqt (Step 3 merged file)')
    print(f'   is used unchanged for Steps 4–7.')

# ── Case 2: nothing entered — clean skip, fully rigid receptor ───────────────
elif not flex_method_a and not flex_method_b:
    print('ℹ️  No flexible residues specified — this is a fully rigid-receptor run.')
    print(f'   {receptor_name}.pdbqt (Step 3 merged file) is used unchanged for Steps 4–7.')

# ── Case 3: a selection was made — attempt prepare_flexreceptor4 ─────────────
else:
    if 'PREPARE_FLEXRECEPTOR4' not in dir() or not PREPARE_FLEXRECEPTOR4:
        print('❌ prepare_flexreceptor4.py was not located during Step 1.')
        print('   Re-run Step 1, or proceed without flexible residues (rigid-only).')
        print(f'\nℹ️  Falling back to rigid-only: {receptor_name}.pdbqt used unchanged for Steps 4–7.')
    else:
        FLEX_RESIDUES = flex_method_a if flex_method_a else flex_method_b
        method_label  = 'A (same-chain groups)' if flex_method_a else 'B (individual residues)'

        # Distinct scratch filenames — never collide with the canonical
        # receptor_name.pdbqt while prepare_flexreceptor4.py is simultaneously
        # reading -r and writing -g/-x.
        rigid_scratch = os.path.join(work_dir, f'_flexprep_{receptor_name}_rigid.pdbqt')
        flex_scratch  = os.path.join(work_dir, f'_flexprep_{receptor_name}_flex.pdbqt')

        print(f'🦴 Running prepare_flexreceptor4.py ...')
        print(f'   Method          : {method_label}')
        print(f'   Input (-r)      : {receptor_name}.pdbqt  (Step 3 merged: protein + retained cofactors)')
        print(f'   Flex residues   : {FLEX_RESIDUES}')

        cmd = [
            PREPARE_FLEXRECEPTOR4,
            '-r', receptor_pdbqt,     # the FINAL MERGED protein+cofactor PDBQT from Step 3
            '-s', FLEX_RESIDUES,
            '-g', rigid_scratch,
            '-x', flex_scratch,
            '-v',
        ]
        result   = subprocess.run(cmd, capture_output=True, text=True, cwd=work_dir)
        stdout_l = result.stdout.lower()

        if 'located  0  residues' in stdout_l or 'located 0 residues' in stdout_l:
            print('\n❌ RESIDUE SELECTION FAILED — 0 residues matched.')
            print('   Most likely causes:')
            print('   1. Selection string does not match a token from the inspector cell above.')
            print(f"   2. Molecule-name prefix is wrong — it must be exactly '{receptor_name}'.")
            print('   3. Wrong chain letter or residue number.')
            print(f'\nℹ️  Falling back to rigid-only: {receptor_name}.pdbqt used unchanged for Steps 4–7.')

        elif result.returncode != 0 or not os.path.exists(rigid_scratch) or os.path.getsize(rigid_scratch) == 0:
            print('\n❌ prepare_flexreceptor4.py failed:')
            print(result.stdout[-1500:])
            print(result.stderr[-1500:])
            if 'IndexError: string index out of range' in result.stderr:
                print('\n💡 Hint: an atom in the merged PDBQT likely has a missing/empty AD4 atom')
                print('   type (columns 78–79) — check retained cofactor formatting from Step 2.5.')
            elif 'ZeroDivisionError' in result.stderr:
                print('\n💡 Hint: two atoms share identical coordinates (unresolved altLoc conformer).')
            print(f'\nℹ️  Falling back to rigid-only: {receptor_name}.pdbqt used unchanged for Steps 4–7.')

        else:
            # ── Success: overwrite the canonical receptor_name.pdbqt with the
            #    rigid-part output, so Steps 4–7 need zero awareness of this
            #    branch — same filename, regardless of skip vs. select. ──────
            shutil.copy2(rigid_scratch, receptor_pdbqt)
            flex_pdbqt = os.path.join(work_dir, f'{receptor_name}_flex.pdbqt')
            shutil.copy2(flex_scratch, flex_pdbqt)
            flexible_residues_defined = True

            # prepare_flexreceptor4.py's rigid-part output does NOT write a
            # TER between the protein and any retained cofactor, nor a
            # trailing TER/END -- re-use the same finalize_pdbqt_chain()
            # helper defined in Step 3 to fix both, and to renumber serials
            # continuously (defensive -- prepare_flexreceptor4.py already
            # renumbers from 1 itself, but this keeps the guarantee in one
            # place rather than split across two ADFRsuite tool behaviors).
            finalize_pdbqt_chain(receptor_pdbqt)
            print(f'\U0001f522 {receptor_name}.pdbqt (rigid part): TER/END normalized, '
                  f'serials renumbered continuously.')

            with open(receptor_pdbqt) as fh:
                rigid_lines = fh.readlines()
            with open(flex_pdbqt) as fh:
                flex_lines = fh.readlines()
            n_rigid_atoms = sum(1 for l in rigid_lines if l.startswith(('ATOM', 'HETATM')))
            n_flex_atoms  = sum(1 for l in flex_lines  if l.startswith(('ATOM', 'HETATM')))
            n_branch      = sum(1 for l in flex_lines  if l.startswith('BRANCH'))

            print('\n✅ prepare_flexreceptor4.py completed successfully.')
            print(f'   {receptor_name}.pdbqt       : {n_rigid_atoms} atoms  (rigid part — OVERWRITES Step 3 merged file)')
            print(f'   {receptor_name}_flex.pdbqt  : {n_flex_atoms} atoms, {n_branch} rotatable bonds (BRANCH records)')
            print(f'\n➡️  Steps 4–7 below will use {receptor_name}.pdbqt (now the rigid part).')
            print(f"   {receptor_name}_flex.pdbqt is reserved for AutoDock-GPU's --flexres flag at docking time.")

        # Clean up scratch files regardless of outcome
        for _f in (rigid_scratch, flex_scratch):
            if os.path.exists(_f):
                os.remove(_f)

print(f'\n📌 flexible_residues_defined = {flexible_residues_defined}')


---
## 🔭 Step 4 — Visualize Receptor (3D)
Interactive 3D viewer with selectable representation styles:
**cartoon · surface · stick · sphere · line · cross**

Change the style and color dropdowns, then **re-run the cell** (Shift+Enter) to update.

---

In [ ]:
#@title ▶ View Receptor in 3D — Interactive Style Selector
#@markdown Choose a **protein** representation style and re-run to update.
#@markdown The viewer supports rotation (drag), zoom (scroll), right-drag (translate).
#@markdown > 💡 **Retained cofactors** (HETATM records) are always rendered as
#@markdown > **ball-and-stick** regardless of protein style — identical to PyMOL's
#@markdown > `show sticks, organic` convention so they are never hidden by cartoon or surface.

# ── Protein representation ────────────────────────────────────────────────────
viz_style = "cartoon" #@param ["cartoon", "surface", "stick", "sphere", "line", "cross"]

#@markdown ---
#@markdown **Protein color scheme:**
viz_color = "spectrum" #@param ["spectrum", "chainHetatm", "ssJmol", "whiteCarbon", "greenCarbon", "cyanCarbon", "magentaCarbon", "white", "black", "red", "green", "blue", "cyan", "magenta", "yellow", "purple", "orange", "pink"]

#@markdown ---
#@markdown **Cofactor color scheme** *(HETATM — hardwired ball-and-stick; `greenCarbon` mimics PyMOL default)*:
cofactor_color = "greenCarbon" #@param ["greenCarbon", "whiteCarbon", "cyanCarbon", "magentaCarbon", "orangeCarbon", "yellowCarbon", "Jmol", "spectrum", "red", "green", "blue", "cyan", "magenta", "yellow", "orange", "white"]

#@markdown ---
#@markdown **Surface opacity** *(only active when protein style = **surface**, 0.1–1.0)*:
surf_opacity = 0.7 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

import py3Dmol, os

with open(receptor_pdbqt) as fh:
    pdbqt_str = fh.read()

_has_cofactor = any(l.startswith('HETATM') for l in pdbqt_str.splitlines())

view = py3Dmol.view(width=900, height=580)
view.addModel(pdbqt_str, 'pdbqt')

# ── Stage aesthetics — charcoal-navy backdrop + crisp silhouette outline ──────
# A dark, neutral backdrop is the convention in publication-grade molecular
# graphics (PyMOL/ChimeraX defaults): it maximises contrast for every cartoon
# colour scheme and keeps ball-and-stick cofactors visually "lifted" off the
# protein. The outline render pass adds a thin silhouette edge around every
# atom/ribbon, which is what gives professional renders their crisp, printable
# look (Fresnel/edge-detection trick — purely cosmetic, no effect on geometry).
view.setBackgroundColor('#0b0e14')
view.setViewStyle({'style': 'outline', 'color': '#000000', 'width': 0.04})

# ── Protein — ATOM records only (hetflag: False) ──────────────────────────────
# Using the hetflag selector ensures cofactors are never overridden by the
# protein style block — each section is rendered independently.
if viz_style == 'cartoon':
    # arrows on β-strands + a touch of extra ribbon thickness reads far more
    # like a textbook figure than the flat default ribbon.
    view.setStyle({'hetflag': False},
                  {'cartoon': {'color': viz_color, 'arrows': True, 'thickness': 0.32}})
elif viz_style == 'surface':
    # Ghost cartoon underneath so surface has a backbone reference for lighting
    view.setStyle({'hetflag': False}, {'cartoon': {'color': 'lightgrey', 'opacity': 0.12}})
    view.addSurface(py3Dmol.VDW,
                    {'opacity': surf_opacity, 'colorscheme': viz_color, 'smoothness': 2},
                    {'hetflag': False})
elif viz_style == 'stick':
    view.setStyle({'hetflag': False}, {'stick': {'colorscheme': viz_color, 'radius': 0.16}})
elif viz_style == 'sphere':
    # 'scale' renders a tasteful CPK ball rather than full overlapping VDW
    # spheres, which on a whole protein quickly turns into a featureless blob.
    view.setStyle({'hetflag': False}, {'sphere': {'colorscheme': viz_color, 'scale': 0.28}})
elif viz_style == 'line':
    view.setStyle({'hetflag': False}, {'line': {'colorscheme': viz_color, 'linewidth': 1.6}})
elif viz_style == 'cross':
    view.setStyle({'hetflag': False}, {'cross': {'colorscheme': viz_color, 'linewidth': 2.4}})

# ── Cofactors — always ball-and-stick, hardwired, user-coloured ───────────────
# Ball-and-stick = thin sticks (bonds) + small spheres (atoms) — the universal
# molecular-graphics convention for displaying non-polymer hetero molecules.
# PyMOL equivalent: show sticks + show spheres with sphere_scale 0.35.
if _has_cofactor:
    view.addStyle({'hetflag': True},
                  {'stick':  {'colorscheme': cofactor_color, 'radius': 0.18}})
    view.addStyle({'hetflag': True},
                  {'sphere': {'colorscheme': cofactor_color, 'radius': 0.36}})

view.zoomTo()
view.show()

print(f'✅ Protein   : {viz_style}  |  color: {viz_color}')
if _has_cofactor:
    print(f'✅ Cofactors : ball-and-stick  |  color: {cofactor_color}  (hardwired — always visible)')
else:
    print('ℹ️  No HETATM records detected — all atoms are standard protein residues.')
if viz_style == 'surface':
    print(f'   Surface opacity: {surf_opacity}  (surface only — does not affect other styles)')
print(f'\n   File: {receptor_pdbqt}')
print('ℹ️  Re-run this cell (Shift+Enter) after changing any dropdown or slider.')


---
## 🔍 Step 5 — Binding Pocket Detection
This cell runs and prints **all detected pockets** in a ranked table.
The table includes pocket rank, score, fill count, center coordinates, radius, and buriedness.

> You will choose which pocket to use in **Step 6** by entering its rank number.
> If you plan to use **Blind docking**, **Manual coordinates**, **Around ligand**, **Around Cofactor** or
> **Around residues** mode, you can skip this step.

---

In [ ]:
#@title ▶ Run Pocket Detection
#@markdown Detects **all** druggable binding pockets on the receptor surface.
#@markdown All pockets are listed in a ranked table — you will select one in the next cell.
#@markdown > ⏭️ You can skip this step if using Blind docking / Manual / Around ligand / Around Cofactor / Around residues mode.

import subprocess, os, re, glob
import numpy as np

autosite_out = os.path.join(work_dir, 'autosite_results')
os.makedirs(autosite_out, exist_ok=True)

cmd = ['autosite', '-r', receptor_pdbqt, '-o', autosite_out]
print('🔍 Running AutoSite pocket detection ...')
result = subprocess.run(cmd, capture_output=True, text=True, cwd=work_dir)

if result.returncode != 0:
    print('❌ AutoSite failed:')
    print(result.stderr[-2000:])
    raise RuntimeError('AutoSite did not complete successfully.')

# ── Step 1: Locate _pockets.txt (search ALL subdirs of work_dir) ──────────────
pocket_files = glob.glob(os.path.join(work_dir, '**', '*_pockets.txt'), recursive=True)
pocket_files += glob.glob(os.path.join(autosite_out, '**', '*_pockets.txt'), recursive=True)
pocket_files = sorted(set(pocket_files))

all_pockets = []  # list of dicts, consumed by Step 6

# ── Step 2: Parse _pockets.txt if found ───────────────────────────────────────
if pocket_files:
    pocket_file = pocket_files[0]
    print(f'\n📋 Pocket file: {pocket_file}\n')
    with open(pocket_file) as fh:
        raw_lines = [l.rstrip() for l in fh if l.strip() and not l.startswith('#')]
    for line in raw_lines:
        if re.match(r'^\s*(rank|#|pocket|clust)', line, re.IGNORECASE):
            continue
        parts = line.split()
        if len(parts) < 7:
            continue
        try:
            pocket = {
                'rank'  : int(parts[0]),
                'score' : float(parts[1]),
                'nfill' : int(parts[2]),
                'cx'    : float(parts[3]),
                'cy'    : float(parts[4]),
                'cz'    : float(parts[5]),
                'radius': float(parts[6]),
            }
            if len(parts) >= 8:
                pocket['buriedness'] = float(parts[7])
            all_pockets.append(pocket)
        except (ValueError, IndexError):
            continue

# ── Step 3: Parse stdout table if _pockets.txt missing or empty ───────────────
if not all_pockets:
    print('⚠️  _pockets.txt not found or empty — parsing stdout table ...')
    stdout_pockets = {}
    in_table = False
    for line in result.stdout.splitlines():
        if re.match(r'\s*-{3,}\+-{3,}', line):
            in_table = True
            continue
        if not in_table:
            continue
        parts = line.split()
        if len(parts) < 7:
            in_table = False
            continue
        try:
            rank = int(parts[0])
            stdout_pockets[rank] = {
                'rank'       : rank,
                'score'      : float(parts[6]),
                'nfill'      : int(parts[2]),
                'radius'     : float(parts[3]),
                'buriedness' : float(parts[5]),
                'cx': 0.0, 'cy': 0.0, 'cz': 0.0,
            }
        except (ValueError, IndexError):
            continue

    # ── Recover cluster centers from AutoSite feature-point PDB files ─────────
    fp_files = glob.glob(os.path.join(work_dir, '**', '*_cl_*_fp.pdb'), recursive=True)
    fp_files += glob.glob(os.path.join(work_dir, '**', '*_cl_*.pdb'), recursive=True)
    fp_files = sorted(set(fp_files))

    centers_found = 0
    for cf in fp_files:
        m = re.search(r'_cl_(\d+)', cf)
        if not m:
            continue
        rank = int(m.group(1))
        if rank not in stdout_pockets:
            continue
        coords = []
        with open(cf) as fh:
            for line in fh:
                if line.startswith(('ATOM', 'HETATM')):
                    try:
                        coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                    except ValueError:
                        pass
        if coords:
            arr = np.array(coords)
            stdout_pockets[rank]['cx'] = float(arr[:, 0].mean())
            stdout_pockets[rank]['cy'] = float(arr[:, 1].mean())
            stdout_pockets[rank]['cz'] = float(arr[:, 2].mean())
            centers_found += 1

    all_pockets = sorted(stdout_pockets.values(), key=lambda p: p['rank'])
    if centers_found < len(all_pockets):
        print(f'⚠️  Centers recovered for {centers_found}/{len(all_pockets)} pockets from cluster files.')
        print('   Pockets with cx=cy=cz=0 will need manual center override in Step 6.')

# ── Print ranked table ────────────────────────────────────────────────────────
if all_pockets:
    print(f'Found {len(all_pockets)} pocket(s):\n')
    header = f'  {"Rank":>4}  {"Score":>8}  {"Fill":>5}  {"CX":>8}  {"CY":>8}  {"CZ":>8}  {"Radius":>7}'
    if 'buriedness' in all_pockets[0]:
        header += f'  {"Buried":>7}'
    print(header)
    print('  ' + '-'*75)
    for p in all_pockets:
        row = (f'  {p["rank"]:>4}  {p["score"]:>8.3f}  {p["nfill"]:>5}  '
               f'{p["cx"]:>8.3f}  {p["cy"]:>8.3f}  {p["cz"]:>8.3f}  {p["radius"]:>7.2f}')
        if 'buriedness' in p:
            row += f'  {p["buriedness"]:>7.3f}'
        print(row)
    print('\n✅ Proceed to Step 6 to select a pocket and define the grid box.')
else:
    print('❌ No pockets could be parsed. Raw stdout below:')
    print(result.stdout[-3000:])

---
## 📦 Step 6 — Define Grid Box (6 Methods)

| Mode | Coordinates used | Use when |
|---|---|---|
| **Pocket selection** | Auto-derived pocket radius + padding | No known ligand |
| **Blind docking** | Entire receptor extent | Exploratory / unknown site |
| **Manual coordinates** | User-entered centre + size | Precise override |
| **Around ligand** | Co-crystallised ligand atoms **only** | Known binding site |
| **Around cofactor** | Retained cofactor atoms **only** | Cofactor-defined active site (FAD, NAD, heme, PLP…) |
| **Around residues** | Cα of user-specified residue list | Binding site known from literature |

> ⚠️ **Around ligand** and **Around cofactor** are mutually exclusive and strictly isolated:
> specifying a ligand name will never pull in cofactor coordinates, and vice versa.
> Retained cofactors are always present in the PDBQT for energy scoring regardless of box mode.

The viewer shows the **cyan grid box**, **yellow sticks** (residues inside box),
**red/orange spheres** (AutoSite pockets), **magenta spheres** (ligand atoms),
**orange spheres** (cofactor atoms), and **lime spheres** (specified residue Cα).

---


In [ ]:
#@title ▶ Define Grid Box
#@markdown ### Grid box definition method:
grid_method = "AutoSite pocket" #@param ["AutoSite pocket", "Blind docking", "Manual coordinates", "Around ligand", "Around cofactor", "Around residues"]

#@markdown ---
#@markdown ### ① AutoSite — pocket ranks to use (comma-separated, e.g. `1` or `1,3`):
pocket_ranks_str = "1" #@param {type:"string"}
#@markdown Padding beyond pocket radii (Å):
pocket_padding = 5.0 #@param {type:"number"}

#@markdown ---
#@markdown ### ② Around ligand — residue name of the co-crystallised ligand:
#@markdown *(Must be a HETATM residue name detected during Upload, e.g. `LIG`, `ATP`, `STI`)*
#@markdown *(Box is computed **only** from ligand atoms)*
ligand_resname = "LIG" #@param {type:"string"}
#@markdown Padding around ligand heavy atoms (Å):
ligand_padding = 5.0 #@param {type:"number"}

#@markdown ---
#@markdown ### ③ Around cofactor — residue name(s) of retained cofactor(s):
#@markdown *(Comma-separated 3-letter codes, e.g. `FAD` or `FAD, NAD`)*
#@markdown *(Box is computed **only** from cofactor atoms)*
cofactor_box_resnames = "" #@param {type:"string"}
#@markdown Padding around cofactor heavy atoms (Å):
cofactor_padding = 5.0 #@param {type:"number"}

#@markdown ---
#@markdown ### ④ Around residues — chain:resnum pairs, comma-separated:
#@markdown e.g. `A:123, A:124, B:200` — or just `123, 124` (defaults to chain A)
user_residues_str = "" #@param {type:"string"}
#@markdown Padding around residue Cα atoms (Å):
residue_padding = 5.0 #@param {type:"number"}

#@markdown ---
#@markdown ### ⑤ Manual / override center coordinates (Å):
center_x = 0.0 #@param {type:"number"}
center_y = 0.0 #@param {type:"number"}
center_z = 0.0 #@param {type:"number"}
#@markdown Grid box size in points (odd recommended) — used for Manual & Blind docking:
size_x = 60 #@param {type:"integer"}
size_y = 60 #@param {type:"integer"}
size_z = 60 #@param {type:"integer"}

#@markdown ---
#@markdown ### Grid spacing (Å):
spacing = 0.375 #@param {type:"number"}

#@markdown ---
#@markdown ### Visualization style for the protein backbone:
box_viz_style = "cartoon" #@param ["cartoon", "surface", "stick", "sphere", "line", "cross"]
box_viz_color = "spectrum" #@param ["spectrum", "chainHetatm", "ssJmol", "whiteCarbon", "greenCarbon", "cyanCarbon", "white", "black", "red", "green", "blue", "cyan", "magenta", "yellow", "purple", "orange", "pink"]
#@markdown **Surface opacity** *(only active when protein style = **surface**)*:
box_surf_opacity = 0.6 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
#@markdown **Cofactor color** *(HETATM records — always ball-and-stick; `greenCarbon` = PyMOL default)*:
box_cofactor_color = "greenCarbon" #@param ["greenCarbon", "whiteCarbon", "cyanCarbon", "magentaCarbon", "orangeCarbon", "yellowCarbon", "Jmol", "spectrum", "red", "green", "blue", "cyan", "magenta", "yellow", "orange", "white"]

import os, math, re
import numpy as np
import py3Dmol
from collections import defaultdict

# ── Parse receptor atoms from prepared PDBQT (Cα-based for residue detection) ─
receptor_atoms = []
ca_atoms = {}          # (chain, resnum) → {resname, x, y, z}

with open(receptor_pdbqt) as fh:
    for line in fh:
        if not line.startswith(('ATOM', 'HETATM')):
            continue
        try:
            atom_name = line[12:16].strip()
            resname   = line[17:20].strip()
            chain     = line[21].strip()
            resnum    = int(line[22:26].strip())
            x, y, z   = float(line[30:38]), float(line[38:46]), float(line[46:54])
        except (ValueError, IndexError):
            continue
        receptor_atoms.append({
            'atom': atom_name, 'resname': resname,
            'chain': chain, 'resnum': resnum,
            'x': x, 'y': y, 'z': z,
        })
        key = (chain, resnum)
        if atom_name == 'CA':
            ca_atoms[key] = {'resname': resname, 'x': x, 'y': y, 'z': z}
        elif atom_name == "C1'" and key not in ca_atoms:
            ca_atoms[key] = {'resname': resname, 'x': x, 'y': y, 'z': z}
        elif key not in ca_atoms:
            ca_atoms[key] = {'resname': resname, 'x': x, 'y': y, 'z': z}

# ── Helper: Cα-based residue list inside an axis-aligned box ─────────────────
def residues_in_box(ca_dict, box_cx, box_cy, box_cz, sx, sy, sz, step):
    hx, hy, hz = (sx*step)/2, (sy*step)/2, (sz*step)/2
    inside = []
    for (chain, resnum), info in ca_dict.items():
        if (abs(info['x']-box_cx) <= hx and
            abs(info['y']-box_cy) <= hy and
            abs(info['z']-box_cz) <= hz):
            inside.append((chain, resnum, info['resname']))
    return sorted(inside, key=lambda t: (t[0], t[1]))

# ── Helper: bounding box + padding → grid center + odd point counts ───────────
def bbox_to_grid(coords_arr, pad, spc):
    lo = coords_arr.min(axis=0) - pad
    hi = coords_arr.max(axis=0) + pad
    c  = (lo + hi) / 2
    sp = hi - lo
    def to_pts(d):
        v = int(math.ceil(d / spc))
        return v + (1 if v % 2 == 0 else 0)
    return float(c[0]), float(c[1]), float(c[2]), to_pts(sp[0]), to_pts(sp[1]), to_pts(sp[2])

# ─────────────────────────────────────────────────────────────────────────────
selected_pockets = []
selected_ranks   = []
parsed_res       = []   # filled by Around residues mode
res_coords       = []   # corresponding Cα xyz
_cofactor_coord_sets = {}   # {resname: np.array} — used for 3D overlay

# ── METHOD 1: AutoSite pocket ─────────────────────────────────────────────────
if grid_method == 'AutoSite pocket':
    if not all_pockets:
        raise RuntimeError('No pockets available — run the AutoSite cell (Step 5) first.')
    try:
        selected_ranks = [int(r.strip()) for r in pocket_ranks_str.split(',') if r.strip()]
    except ValueError:
        raise ValueError(f'Invalid pocket_ranks_str: "{pocket_ranks_str}". Use comma-separated integers.')
    rank_map = {p['rank']: p for p in all_pockets}
    missing  = [r for r in selected_ranks if r not in rank_map]
    if missing:
        raise ValueError(f'Pocket rank(s) {missing} not found. Available: {sorted(rank_map.keys())}')
    selected_pockets = [rank_map[r] for r in selected_ranks]

    mins = np.array([[p['cx']-p['radius'], p['cy']-p['radius'], p['cz']-p['radius']] for p in selected_pockets])
    maxs = np.array([[p['cx']+p['radius'], p['cy']+p['radius'], p['cz']+p['radius']] for p in selected_pockets])
    box_min = mins.min(axis=0) - pocket_padding
    box_max = maxs.max(axis=0) + pocket_padding
    cx = float((box_min[0]+box_max[0])/2)
    cy = float((box_min[1]+box_max[1])/2)
    cz = float((box_min[2]+box_max[2])/2)
    span = box_max - box_min
    def to_pts(d): v = int(math.ceil(d/spacing)); return v+(1 if v%2==0 else 0)
    size_x, size_y, size_z = to_pts(span[0]), to_pts(span[1]), to_pts(span[2])

    print(f'✅ {len(selected_pockets)} pocket(s) selected: {selected_ranks}')
    for p in selected_pockets:
        print(f'  Pocket #{p["rank"]}  score={p["score"]:.3f}  fill={p["nfill"]}  '
              f'center=({p["cx"]:.2f},{p["cy"]:.2f},{p["cz"]:.2f})  r={p["radius"]:.2f} Å')
    print(f'\n  Combined center : ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
    print(f'  Combined size   : {size_x} x {size_y} x {size_z} pts')
    print('\n📋 All pockets:')
    print(f'  {"Rank":>4}  {"Score":>8}  {"CX":>8}  {"CY":>8}  {"CZ":>8}  {"R":>6}  {"Buried":>7}  Status')
    print('  ' + '-'*72)
    for p in all_pockets:
        buried_str = f'{p["buriedness"]:.3f}' if 'buriedness' in p else '  -   '
        status = '  ◀ SELECTED' if p['rank'] in selected_ranks else ''
        print(f'  {p["rank"]:>4}  {p["score"]:>8.3f}  {p["cx"]:>8.3f}  '
              f'{p["cy"]:>8.3f}  {p["cz"]:>8.3f}  {p["radius"]:>6.2f}  {buried_str:>7}{status}')

# ── METHOD 2: Blind docking ───────────────────────────────────────────────────
elif grid_method == 'Blind docking':
    coords = np.array([[a['x'], a['y'], a['z']] for a in receptor_atoms])
    cx, cy, cz = [float(v) for v in coords.mean(axis=0)]
    span = coords.max(axis=0) - coords.min(axis=0)
    def to_pts(d): v = int(np.ceil(d/spacing))+10; return v+(1 if v%2==0 else 0)
    size_x, size_y, size_z = to_pts(span[0]), to_pts(span[1]), to_pts(span[2])
    print(f'🌐 Blind docking — receptor geometric centre:')
    print(f'   Receptor span : {span[0]:.1f} x {span[1]:.1f} x {span[2]:.1f} Å')
    print(f'   Centre        : ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
    print(f'   Box size      : {size_x} x {size_y} x {size_z} pts')

# ── METHOD 3: Manual coordinates ─────────────────────────────────────────────
elif grid_method == 'Manual coordinates':
    cx, cy, cz = center_x, center_y, center_z
    print(f'📌 Manual coordinates: ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
    print(f'   Box : {size_x} x {size_y} x {size_z} pts')

# ── METHOD 4: Around ligand ───────────────────────────────────────────────────
# Box is computed EXCLUSIVELY from the named ligand's heavy atom coordinates.
# Retained cofactors (FAD, NAD, NAG, etc.) are present in the PDBQT for
# correct energy scoring but are intentionally excluded from the box calculation.
# Use 'Around cofactor' if you want the box centred on a cofactor instead.
elif grid_method == 'Around ligand':
    _lstore = ligand_coords_store if 'ligand_coords_store' in dir() else {}
    _rname  = ligand_resname.strip().upper()
    if not _lstore:
        raise RuntimeError(
            'ligand_coords_store is empty. Re-run the Upload PDB cell (Step 2) — '
            'ligand coordinates must be captured before prepare_receptor strips them.'
        )
    if _rname not in _lstore:
        avail = sorted(_lstore.keys())
        raise ValueError(
            f"Ligand '{_rname}' not found in stored HETATM records.\n"
            f"Available residue names: {avail}\n"
            f"Check the table printed in Step 2 and set ligand_resname accordingly."
        )

    # Strictly ligand-only coordinates — no cofactor merging
    lig_coords = np.array(_lstore[_rname]['coords'])
    cx, cy, cz, size_x, size_y, size_z = bbox_to_grid(lig_coords, ligand_padding, spacing)

    lig_span = lig_coords.max(axis=0) - lig_coords.min(axis=0)
    print(f'✅ Grid box around ligand: {_rname}  (ligand atoms only)')
    print(f'   Atoms         : {len(lig_coords)}')
    print(f'   Ligand span   : {lig_span[0]:.2f} x {lig_span[1]:.2f} x {lig_span[2]:.2f} Å')
    print(f'   Padding       : {ligand_padding} Å')
    print(f'   Box centre    : ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
    print(f'   Box size      : {size_x} x {size_y} x {size_z} pts')

    # Inform user if cofactors were retained but intentionally excluded from box
    _retained = cofactors_to_retain if 'cofactors_to_retain' in dir() else set()
    _excluded = _retained - {_rname}
    if _excluded:
        print(f'\n   ℹ️  Retained cofactors present in PDBQT but excluded from box: {sorted(_excluded)}')
        print(f'      → Use "Around cofactor" mode to centre the box on a cofactor instead.')
    print(f'\n   ℹ️  Cofactor atoms contribute to AutoGrid4 energy maps (charges + vdW),')
    print(f'      but the box boundary is determined by ligand coordinates only.')

# ── METHOD 5: Around cofactor ─────────────────────────────────────────────────
# Box is computed EXCLUSIVELY from the named cofactor(s) heavy atom coordinates.
# The docking ligand coordinates are NOT included in the bounding box.
# Use this when the cofactor defines the binding site (FAD in flavoenzymes,
# NAD+ as co-substrate, heme in P450s, PLP in PLP-dependent enzymes, etc.).
elif grid_method == 'Around cofactor':
    _lstore = ligand_coords_store if 'ligand_coords_store' in dir() else {}
    if not _lstore:
        raise RuntimeError(
            'ligand_coords_store is empty. Re-run the Upload PDB cell (Step 2).'
        )

    # Parse cofactor residue name(s) from user input
    _cof_input = cofactor_box_resnames.strip().upper()
    if not _cof_input:
        # Fall back to the full retained set if field is left blank
        _retained = cofactors_to_retain if 'cofactors_to_retain' in dir() else set()
        if not _retained:
            raise ValueError(
                'cofactor_box_resnames is empty and no cofactors were retained in Step 2.5.\n'
                'Enter a residue name (e.g. "FAD") or run Step 2.5 first.'
            )
        requested_cofs = _retained
        print(f'ℹ️  cofactor_box_resnames left blank — using all retained cofactors: {sorted(requested_cofs)}')
    else:
        requested_cofs = set(tok.strip() for tok in re.split(r'[,\s]+', _cof_input) if tok.strip())

    # Validate each requested cofactor has stored coordinates
    found_cofs   = requested_cofs & set(_lstore.keys())
    missing_cofs = requested_cofs - found_cofs
    if missing_cofs:
        avail = sorted(_lstore.keys())
        print(f'⚠️  Cofactor(s) not found in stored HETATM records (ignored): {sorted(missing_cofs)}')
        print(f'   Available residue names: {avail}')
    if not found_cofs:
        raise ValueError(
            f'None of the requested cofactors {sorted(requested_cofs)} were found.\n'
            f'Available: {sorted(_lstore.keys())}'
        )

    # Collect all heavy-atom coordinates for the requested cofactor(s)
    all_cof_coords_list = []
    for _cf_rn in sorted(found_cofs):
        _cf_arr = np.array(_lstore[_cf_rn]['coords'])
        all_cof_coords_list.append(_cf_arr)
        _cofactor_coord_sets[_cf_rn] = _cf_arr   # for 3D overlay

    all_cof_coords = np.vstack(all_cof_coords_list)
    cx, cy, cz, size_x, size_y, size_z = bbox_to_grid(all_cof_coords, cofactor_padding, spacing)

    cof_span = all_cof_coords.max(axis=0) - all_cof_coords.min(axis=0)
    print(f'✅ Grid box around cofactor(s): {sorted(found_cofs)}  (cofactor atoms only)')
    for _cf_rn in sorted(found_cofs):
        print(f'   {_cf_rn}: {len(_lstore[_cf_rn]["coords"])} atoms')
    print(f'   Combined span : {cof_span[0]:.2f} x {cof_span[1]:.2f} x {cof_span[2]:.2f} Å')
    print(f'   Padding       : {cofactor_padding} Å')
    print(f'   Box centre    : ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
    print(f'   Box size      : {size_x} x {size_y} x {size_z} pts')

    # Containment check — verify all cofactor atoms fall inside the computed box
    hx, hy, hz = (size_x*spacing)/2, (size_y*spacing)/2, (size_z*spacing)/2
    print(f'\n   📐 Containment check (box ±{hx:.2f} x ±{hy:.2f} x ±{hz:.2f} Å from centre):')
    for _cf_rn, _cf_arr in _cofactor_coord_sets.items():
        n_in = int(np.sum(
            (np.abs(_cf_arr[:,0]-cx) <= hx) &
            (np.abs(_cf_arr[:,1]-cy) <= hy) &
            (np.abs(_cf_arr[:,2]-cz) <= hz)
        ))
        status = '✅' if n_in == len(_cf_arr) else '⚠️ '
        print(f'   {status} {_cf_rn}: {n_in}/{len(_cf_arr)} atoms inside box')

    # Inform user about docking ligand being excluded
    _lig_rn = ligand_resname.strip().upper()
    if _lig_rn in _lstore and _lig_rn not in found_cofs:
        print(f'\n   ℹ️  Ligand "{_lig_rn}" coordinates are NOT included in this box.')
        print(f'      → Use "Around ligand" mode to centre the box on the ligand instead.')

# ── METHOD 6: Around residues ─────────────────────────────────────────────────
elif grid_method == 'Around residues':
    if not user_residues_str.strip():
        raise ValueError(
            "user_residues_str is empty.\n"
            "Enter residues as 'A:123, A:124' (chain:resnum) or '123, 124' (defaults to chain A)."
        )
    parsed_res = []
    for token in user_residues_str.split(','):
        token = token.strip()
        if not token:
            continue
        if ':' in token:
            ch, rn = token.split(':', 1)
            parsed_res.append((ch.strip().upper(), int(rn.strip())))
        else:
            parsed_res.append(('A', int(token.strip())))

    if not parsed_res:
        raise ValueError('Could not parse any residues from user_residues_str.')

    res_coords    = []
    not_found_res = []
    for ch, rn in parsed_res:
        key = (ch, rn)
        if key in ca_atoms:
            info = ca_atoms[key]
            res_coords.append((info['x'], info['y'], info['z']))
        else:
            found = False
            for (kch, krn), info in ca_atoms.items():
                if krn == rn:
                    res_coords.append((info['x'], info['y'], info['z']))
                    found = True
                    break
            if not found:
                not_found_res.append(f'{ch}:{rn}')

    if not_found_res:
        print(f'⚠️  Residues not found in receptor PDBQT: {not_found_res}')
        print('   These residues were skipped. Check residue numbering and chain IDs.')
    if not res_coords:
        raise ValueError('None of the specified residues were found. Check numbers and chains.')

    coords_arr = np.array(res_coords)
    cx, cy, cz, size_x, size_y, size_z = bbox_to_grid(coords_arr, residue_padding, spacing)

    print(f'✅ Grid box around {len(res_coords)} residue(s) (Cα-based):')
    for i, (ch, rn) in enumerate(parsed_res):
        if i >= len(res_coords):
            break
        rname = ca_atoms.get((ch, rn), {}).get('resname', '???')
        x, y, z = res_coords[i]
        print(f'   {ch}:{rn:>4} ({rname})  Cα = ({x:.2f}, {y:.2f}, {z:.2f}) Å')
    print(f'   Padding   : {residue_padding} Å')
    print(f'   Centre    : ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
    print(f'   Box size  : {size_x} x {size_y} x {size_z} pts')

npts = (size_x, size_y, size_z)

# ── Final grid summary ────────────────────────────────────────────────────────
print(f'\n📦 Final Grid Box:')
print(f'   Centre  : ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
print(f'   Size    : {size_x} x {size_y} x {size_z} points')
print(f'   Spacing : {spacing} Å')
print(f'   Volume  : {size_x*spacing:.2f} x {size_y*spacing:.2f} x {size_z*spacing:.2f} Å')

# ── Residues inside the box (Cα-based) ───────────────────────────────────────
box_residues = residues_in_box(ca_atoms, cx, cy, cz, size_x, size_y, size_z, spacing)
print(f'\n🧬 Residues inside the grid box: {len(box_residues)} residues')
if box_residues:
    by_chain = defaultdict(list)
    for chain, resnum, resname in box_residues:
        by_chain[chain].append((resnum, resname))
    for chain, residues in sorted(by_chain.items()):
        chain_label = chain if chain else '?'
        resnum_list = ', '.join(f'{rn}({rm})' for rn, rm in residues)
        print(f'  Chain {chain_label}: {resnum_list}')
    all_resnums = sorted(set(rn for _, rn, _ in box_residues))
    print(f'\n  Residue numbers only: {all_resnums}')
else:
    print('  ⚠️  No residues found inside the box — check centre / size.')

# ── 3D visualisation ─────────────────────────────────────────────────────────
# Design philosophy:
#   • Protein style is always user-controlled via box_viz_style / box_viz_color.
#   • For "Around ligand" / "Around cofactor" modes the protein is additionally
#     dimmed with a translucent white surface overlay so the focal molecule
#     draws the eye — identical to PyMOL's "protein in bg, ligand in fg" look.
#   • Cofactors (HETATM) are always ball-and-stick regardless of protein style.
#   • Each method gets its own colour-coded overlay (pocket spheres, ligand
#     spheres, cofactor spheres, residue Cα markers).
#   • The grid box is rendered as a solid cyan wireframe + barely-visible fill
#     with a red centre sphere and a labelled dimension annotation.
# ─────────────────────────────────────────────────────────────────────────────
with open(receptor_pdbqt) as fh:
    pdbqt_str = fh.read()

_has_cofactor = any(l.startswith('HETATM') for l in pdbqt_str.splitlines())
_dim_protein  = grid_method in ('Around ligand', 'Around cofactor')

view = py3Dmol.view(width=900, height=600)
view.addModel(pdbqt_str, 'pdbqt')

# ── Stage aesthetics — charcoal-navy backdrop + crisp silhouette outline ──────
# Dark neutral backdrop maximises contrast against every protein colour scheme
# and against the cyan grid box; the outline pass gives every ribbon/atom a
# thin edge so the rendered scene reads like a publication figure rather than
# a flat default viewport. Purely cosmetic — geometry and coordinates below
# are untouched.
view.setBackgroundColor('#0b0e14')
view.setViewStyle({'style': 'outline', 'color': '#000000', 'width': 0.04})

# ── Protein base style (ATOM records only — hetflag: False) ───────────────────
if box_viz_style == 'cartoon':
    view.setStyle({'hetflag': False},
                  {'cartoon': {'color': box_viz_color, 'arrows': True, 'thickness': 0.32}})
elif box_viz_style == 'surface':
    view.setStyle({'hetflag': False}, {'cartoon': {'color': 'lightgrey', 'opacity': 0.12}})
    view.addSurface(py3Dmol.VDW,
                    {'opacity': box_surf_opacity, 'colorscheme': box_viz_color, 'smoothness': 2},
                    {'hetflag': False})
elif box_viz_style == 'stick':
    view.setStyle({'hetflag': False}, {'stick': {'colorscheme': box_viz_color, 'radius': 0.16}})
elif box_viz_style == 'sphere':
    view.setStyle({'hetflag': False}, {'sphere': {'colorscheme': box_viz_color, 'scale': 0.28}})
elif box_viz_style == 'line':
    view.setStyle({'hetflag': False}, {'line': {'colorscheme': box_viz_color, 'linewidth': 1.6}})
elif box_viz_style == 'cross':
    view.setStyle({'hetflag': False}, {'cross': {'colorscheme': box_viz_color, 'linewidth': 2.4}})

# For ligand/cofactor-centric modes, overlay a dim white surface on the protein
# (even over the already-applied style) so the focal molecule stands out.
# PyMOL equivalent: set_color bg_protein, [0.85,0.85,0.85]; set transparency 0.7
if _dim_protein and box_viz_style != 'surface':
    view.addSurface(py3Dmol.VDW,
                    {'opacity': 0.22, 'colorscheme': 'whiteCarbon', 'smoothness': 2},
                    {'hetflag': False})

# ── Retained cofactors — always ball-and-stick (hetflag: True) ────────────────
if _has_cofactor:
    view.addStyle({'hetflag': True},
                  {'stick':  {'colorscheme': box_cofactor_color, 'radius': 0.18}})
    view.addStyle({'hetflag': True},
                  {'sphere': {'colorscheme': box_cofactor_color, 'radius': 0.36}})

# ── Residues with Cα inside the box — yellow sticks ──────────────────────────
# PyMOL equivalent: select box_res, (protein within <Å> of grid_centre); show sticks, box_res
for _ch, _rn, _ in box_residues:
    view.addStyle({'resi': _rn, 'chain': _ch, 'hetflag': False},
                  {'stick': {'colorscheme': 'yellowCarbon', 'radius': 0.20}})

# ── Method-specific overlays ──────────────────────────────────────────────────

# ── AutoSite pocket: translucent spheres + solid centre dots + labels ─────────
if grid_method == 'AutoSite pocket' and all_pockets:
    for p in all_pockets:
        _sel = p['rank'] in selected_ranks
        _col = 'red' if _sel else 'orange'
        # Translucent volume sphere (pocket extent)
        view.addSphere({
            'center':    {'x': p['cx'], 'y': p['cy'], 'z': p['cz']},
            'radius':    p['radius'],
            'color':     _col,
            'opacity':   0.30 if _sel else 0.15,
            'wireframe': False,
        })
        # Solid centre marker
        view.addSphere({
            'center':  {'x': p['cx'], 'y': p['cy'], 'z': p['cz']},
            'radius':  0.55,
            'color':   _col,
            'opacity': 0.95,
        })
        # Label — larger + bold for selected pockets
        view.addLabel(
            f'P{p["rank"]}  score={p["score"]:.2f}',
            {'position':         {'x': p['cx'], 'y': p['cy'] + p['radius'] + 0.8, 'z': p['cz']},
             'fontColor':        'white' if _sel else '#ffcc44',
             'fontSize':         13 if _sel else 11,
             'fontStyle':        'bold' if _sel else 'normal',
             'fontFamily':       'Helvetica',
             'backgroundColor':  '#15171f',
             'backgroundOpacity':0.80,
             'borderColor':      _col,
             'borderThickness':  1.2 if _sel else 0.6,
             'borderOpacity':    0.95,
             'padding':          5,
             'alignment':        'center'}
        )

# ── Blind docking: mark the geometric centre ──────────────────────────────────
elif grid_method == 'Blind docking':
    view.addSphere({'center': {'x': cx, 'y': cy, 'z': cz},
                    'radius': 1.4, 'color': 'cyan', 'opacity': 0.85})
    view.addLabel(
        f'Geometric centre  ({cx:.1f}, {cy:.1f}, {cz:.1f}) Å',
        {'position':         {'x': cx, 'y': cy + (size_y * spacing / 2) + 3.5, 'z': cz},
         'fontColor':        '#00ffff',
         'fontSize':         12,
         'fontStyle':        'bold',
         'fontFamily':       'Helvetica',
         'backgroundColor':  '#0d1117',
         'backgroundOpacity':0.80,
         'borderColor':      'cyan',
         'borderThickness':  0.8,
         'borderOpacity':    0.95,
         'padding':          5,
         'alignment':        'center'}
    )

# ── Manual coordinates: prominent magenta centre marker ───────────────────────
elif grid_method == 'Manual coordinates':
    view.addSphere({'center': {'x': cx, 'y': cy, 'z': cz},
                    'radius': 1.1, 'color': 'magenta', 'opacity': 0.95})
    view.addLabel(
        f'Manual centre  ({cx:.2f}, {cy:.2f}, {cz:.2f}) Å',
        {'position':         {'x': cx, 'y': cy + (size_y * spacing / 2) + 3.0, 'z': cz},
         'fontColor':        '#ff44ff',
         'fontSize':         12,
         'fontStyle':        'bold',
         'fontFamily':       'Helvetica',
         'backgroundColor':  '#0d1117',
         'backgroundOpacity':0.80,
         'borderColor':      'magenta',
         'borderThickness':  0.8,
         'borderOpacity':    0.95,
         'padding':          5,
         'alignment':        'center'}
    )

# ── Around ligand: magenta spheres at crystal coords + label ──────────────────
elif grid_method == 'Around ligand' and 'ligand_coords_store' in dir():
    _rname_l = ligand_resname.strip().upper()
    if _rname_l in ligand_coords_store:
        _lcoords = ligand_coords_store[_rname_l]['coords']
        _lc      = np.array(_lcoords).mean(axis=0)
        # Ball-and-stick representation from crystal coordinates
        for (_lx, _ly, _lz) in _lcoords:
            view.addSphere({'center': {'x': _lx, 'y': _ly, 'z': _lz},
                            'radius': 0.46, 'color': 'magenta', 'opacity': 0.95})
        view.addLabel(
            f'{_rname_l}  ({len(_lcoords)} atoms)',
            {'position':         {'x': float(_lc[0]), 'y': float(_lc[1]) + 2.5, 'z': float(_lc[2])},
             'fontColor':        '#ff77ff',
             'fontSize':         14,
             'fontStyle':        'bold',
             'fontFamily':       'Helvetica',
             'backgroundColor':  '#0d1117',
             'backgroundOpacity':0.82,
             'borderColor':      'magenta',
             'borderThickness':  1.4,
             'borderOpacity':    0.95,
             'padding':          6,
             'alignment':        'center'}
        )

# ── Around cofactor: per-cofactor coloured spheres + labels ───────────────────
# Each retained cofactor in _cofactor_coord_sets gets its own accent colour,
# cycling through a palette that is distinct from protein/box colours.
elif grid_method == 'Around cofactor' and _cofactor_coord_sets:
    _cof_palette = ['#00ff88', '#ff8c00', '#00ccff', '#ff44aa', '#ffff44']
    for _cidx, (_cf_rn, _cf_arr) in enumerate(_cofactor_coord_sets.items()):
        _ccol  = _cof_palette[_cidx % len(_cof_palette)]
        _ccent = _cf_arr.mean(axis=0)
        # Highlight spheres (larger than base ball-and-stick) at crystal coords
        for (_cfx, _cfy, _cfz) in _cf_arr:
            view.addSphere({'center': {'x': float(_cfx), 'y': float(_cfy), 'z': float(_cfz)},
                            'radius': 0.55, 'color': _ccol, 'opacity': 0.90})
        view.addLabel(
            f'{_cf_rn}  ({len(_cf_arr)} atoms)',
            {'position':         {'x': float(_ccent[0]),
                                   'y': float(_ccent[1]) + 2.5,
                                   'z': float(_ccent[2])},
             'fontColor':        _ccol,
             'fontSize':         14,
             'fontStyle':        'bold',
             'fontFamily':       'Helvetica',
             'backgroundColor':  '#0d1117',
             'backgroundOpacity':0.82,
             'borderColor':      _ccol,
             'borderThickness':  1.4,
             'borderOpacity':    0.95,
             'padding':          6,
             'alignment':        'center'}
        )

# ── Around residues: orange sticks + lime Cα spheres + per-residue labels ─────
elif grid_method == 'Around residues' and parsed_res and res_coords:
    for (_ch_r, _rn_r), (_rx, _ry, _rz) in zip(parsed_res, res_coords):
        _rname_r = ca_atoms.get((_ch_r, _rn_r), {}).get('resname', '???')
        # Prominent sticks + spheres for the specified residue
        view.addStyle({'resi': _rn_r, 'chain': _ch_r, 'hetflag': False},
                      {'stick':  {'colorscheme': 'orangeCarbon', 'radius': 0.24}})
        view.addStyle({'resi': _rn_r, 'chain': _ch_r, 'hetflag': False},
                      {'sphere': {'colorscheme': 'orangeCarbon', 'radius': 0.28}})
        # Lime Cα marker sphere
        view.addSphere({'center': {'x': _rx, 'y': _ry, 'z': _rz},
                        'radius': 0.58, 'color': 'lime', 'opacity': 0.95})
        # Per-residue label
        view.addLabel(
            f'{_rname_r}{_rn_r}',
            {'position':         {'x': _rx, 'y': _ry + 1.8, 'z': _rz},
             'fontColor':        '#ccff44',
             'fontSize':         11,
             'fontStyle':        'bold',
             'fontFamily':       'Helvetica',
             'backgroundColor':  '#0d1117',
             'backgroundOpacity':0.70,
             'borderColor':      'lime',
             'borderThickness':  0.7,
             'borderOpacity':    0.95,
             'padding':          4,
             'alignment':        'center'}
        )

# ── Grid box — luminous glass-box rendering ───────────────────────────────────
# Two nested wireframes (outer faint cyan + inner crisp cyan) plus a soft fill
# give the box a "glass panel" depth cue instead of a flat single outline,
# while the docking-relevant geometry (centre, size, spacing) is unchanged.
_bdims = {'w': size_x * spacing, 'h': size_y * spacing, 'd': size_z * spacing}
# Barely-visible fill so interior volume is perceptible
view.addBox({'center': {'x': cx, 'y': cy, 'z': cz},
             'dimensions': _bdims, 'color': 'cyan', 'opacity': 0.06, 'wireframe': False})
# Soft outer halo wireframe (slightly larger, very faint) for a glow effect
_bdims_halo = {'w': _bdims['w'] * 1.01, 'h': _bdims['h'] * 1.01, 'd': _bdims['d'] * 1.01}
view.addBox({'center': {'x': cx, 'y': cy, 'z': cz},
             'dimensions': _bdims_halo, 'color': 'cyan', 'opacity': 0.25, 'wireframe': True})
# Solid crisp cyan wireframe — the true box boundary
view.addBox({'center': {'x': cx, 'y': cy, 'z': cz},
             'dimensions': _bdims, 'color': 'cyan', 'opacity': 1.0,  'wireframe': True})
# Red centre sphere
view.addSphere({'center': {'x': cx, 'y': cy, 'z': cz},
                'radius': 0.45, 'color': 'red', 'opacity': 0.95})
# Dimension label floating above the top face of the box
view.addLabel(
    f'{size_x}×{size_y}×{size_z} pts  |  {size_x*spacing:.1f}×{size_y*spacing:.1f}×{size_z*spacing:.1f} Å',
    {'position':         {'x': cx, 'y': cy + (_bdims['h'] / 2) + 2.2, 'z': cz},
     'fontColor':        '#00ffff',
     'fontSize':         13,
     'fontStyle':        'bold',
     'fontFamily':       'Helvetica',
     'backgroundColor':  '#0a0c10',
     'backgroundOpacity':0.85,
     'borderColor':      'cyan',
     'borderThickness':  1.1,
     'borderOpacity':    0.95,
     'padding':          6,
     'alignment':        'center'}
)

view.zoomTo()
view.show()

# ── Legend ───────────────────────────────────────────────────────────────────
print(f'\n🔭 Visualization Legend  [{grid_method}]')
print(f'   ─────────────────────────────────────────────────────')
print(f'   Cyan wireframe box   : grid box  ({size_x}×{size_y}×{size_z} pts | ')
print(f'                          {size_x*spacing:.1f}×{size_y*spacing:.1f}×{size_z*spacing:.1f} Å)')
print(f'   Red sphere           : grid centre  ({cx:.2f}, {cy:.2f}, {cz:.2f}) Å')
if box_residues:
    print(f'   Yellow sticks        : {len(box_residues)} residues with Cα inside the box')
if _has_cofactor:
    print(f'   Ball-and-stick (bg)  : retained cofactors (HETATM)  — always visible')
if grid_method == 'AutoSite pocket':
    print(f'   Red sphere (volume)  : selected AutoSite pocket(s) {selected_ranks}')
    print(f'   Orange spheres       : unselected AutoSite pockets')
elif grid_method == 'Blind docking':
    print(f'   Cyan sphere          : receptor geometric centre')
elif grid_method == 'Manual coordinates':
    print(f'   Magenta sphere       : user-defined centre ({cx:.2f}, {cy:.2f}, {cz:.2f}) Å')
elif grid_method == 'Around ligand':
    print(f'   Magenta spheres      : {ligand_resname.strip().upper()} crystal atoms (stripped from PDBQT; displayed from Step 2 store)')
    print(f'   ⚠️  Protein dimmed via translucent overlay to highlight the ligand.')
elif grid_method == 'Around cofactor':
    print(f'   Coloured spheres     : cofactor highlight atoms ({", ".join(_cofactor_coord_sets.keys())})')
    print(f'   ⚠️  Protein dimmed via translucent overlay to highlight cofactor(s).')
elif grid_method == 'Around residues':
    print(f'   Orange sticks        : user-specified residues ({len(parsed_res)} residues)')
    print(f'   Lime Cα spheres      : Cα positions of specified residues')



---
## 🗺️ Step 7 — Generate AutoGrid4 Receptor Maps
Writes the Grid Parameter File (`.gpf`) and runs to generate all atom-type affinity maps.
This produces the full set of files required by AutoDock-GPU.

> ⏳ Map generation takes **2–10 minutes** depending on box size and number of atom types.

---

In [ ]:
#@title ▶ Generate GPF & Run AutoGrid4
#@markdown Writes `.gpf`
#@markdown and runs to produce the complete set of receptor map files.

import os, subprocess, shutil

# ── Complete AD4 atom type list ───────────────────────────────────────────────
LIGAND_TYPES = [
    'H', 'HD', 'HS',
    'C', 'A',
    'N', 'NA', 'NS',
    'OA', 'OS',
    'SA', 'S', 'P', 'F',
    'Cl', 'Br', 'I',
    'Mg', 'Ca', 'Mn', 'Fe', 'Zn',
    'G', 'GA', 'J', 'Q', 'Z'
]
# 27 ligand-type maps + e + d = 29 map files
# + gpf, glg, maps.fld, maps.xyz, AD4.1_bound.dat, {receptor_name}.pdbqt = 6
# Grand total = 35 files (single-name scheme — no more duplicate protein/receptor pdbqt)
# (+1 = 36 if Step 3.5 also produced a separate {receptor_name}_flex.pdbqt)

# ── Parse receptor atom types from PDBQT ─────────────────────────────────────
receptor_types = set()
with open(receptor_pdbqt) as fh:
    for line in fh:
        if line.startswith(('ATOM', 'HETATM')):
            at = line[77:79].strip()
            if at:
                receptor_types.add(at)
receptor_types_str = ' '.join(sorted(receptor_types)) if receptor_types else 'A C HD N NA OA SA'

# ── Copy AD4.1_bound.dat to working dir ───────────────────────────────────────
dat_src  = os.environ.get('AD4_DAT', '')
dat_dest = os.path.join(work_dir, 'AD4.1_bound.dat')
if dat_src and os.path.exists(dat_src) and not os.path.exists(dat_dest):
    shutil.copy2(dat_src, dat_dest)
    print(f'✅ Copied AD4.1_bound.dat to working directory')
elif os.path.exists(dat_dest):
    print(f'ℹ️  AD4.1_bound.dat already present')
else:
    print('⚠️  AD4.1_bound.dat not found — autogrid4 may fail.')

# ── Write GPF ─────────────────────────────────────────────────────────────────
gpf_path     = os.path.join(work_dir, f'{receptor_name}.gpf')
glg_path     = os.path.join(work_dir, f'{receptor_name}.glg')
rec_basename = f'{receptor_name}.pdbqt'

gpf_lines = [
    f'npts {npts[0]} {npts[1]} {npts[2]}',
    f'parameter_file AD4.1_bound.dat',
    f'gridfld {receptor_name}.maps.fld',
    f'spacing {spacing}',
    f'receptor_types {receptor_types_str}',
    f'ligand_types {" ".join(LIGAND_TYPES)}',
    f'receptor {rec_basename}',
    f'gridcenter {cx:.3f} {cy:.3f} {cz:.3f}',
    f'smooth 0.5',
]
for at in LIGAND_TYPES:
    gpf_lines.append(f'map {receptor_name}.{at}.map')
gpf_lines += [
    f'elecmap {receptor_name}.e.map',
    f'dsolvmap {receptor_name}.d.map',
    f'dielectric -0.1465',
]

with open(gpf_path, 'w') as fh:
    fh.write('\n'.join(gpf_lines) + '\n')
print(f'✅ GPF written : {gpf_path}')
print(f'   Receptor   : {rec_basename}')
print(f'   Map prefix : {receptor_name}')
print(f'   Atom types : {len(LIGAND_TYPES)} ligand types + e + d = {len(LIGAND_TYPES)+2} map files')
print(f'   npts       : {npts[0]} x {npts[1]} x {npts[2]}  |  spacing: {spacing} Å')
print(f'   Grid centre: ({cx:.3f}, {cy:.3f}, {cz:.3f}) Å')
print(f'   Grid method: {grid_method}\n')

# ── Run autogrid4 ─────────────────────────────────────────────────────────────
print('🗺️  Running autogrid4 ...')
result = subprocess.run(
    ['autogrid4', '-p', gpf_path, '-l', glg_path],
    capture_output=True, text=True, cwd=work_dir
)

if result.returncode == 0:
    map_files   = sorted([f for f in os.listdir(work_dir) if f.endswith('.map')])
    other_files = sorted([f for f in os.listdir(work_dir)
                          if f.endswith(('.fld', '.xyz', '.glg', '.gpf', '.dat', '.pdbqt'))])
    total = len(map_files) + len(other_files)
    expected_total = 35 + (1 if globals().get('flexible_residues_defined', False) else 0)
    print(f'\n✅ autogrid4 completed!')
    print(f'\n   Map files ({len(map_files)}):')
    for f in map_files:
        print(f'   └─ {f}')
    print(f'\n   Supporting files ({len(other_files)}):')
    for f in other_files:
        print(f'   └─ {f}')
    print(f'\n   📦 Total: {total} files')
    if total < expected_total:
        print(f'   ⚠️  Expected {expected_total} files — check .glg log below for missing types.')
    else:
        print(f'   ✅ All {expected_total} files accounted for!')
else:
    print('❌ autogrid4 encountered errors:')
    print(result.stdout[-2000:])
    print(result.stderr[-1000:])
    print('\nTip: Open the .glg file for a detailed error log.')

# ── Tail of log file ──────────────────────────────────────────────────────────
if os.path.exists(glg_path):
    with open(glg_path) as fh:
        log_lines = fh.readlines()
    print('\n📋 Last 15 lines of autogrid4 log:')
    print(''.join(log_lines[-15:]))


---
## 📥 Step 8 — Download Receptor Files
Packages all generated files into a single archive and downloads it to your computer.
Extract the archive and place the contents in your desied folder, when running ADGPU on Google Colab.

---


In [ ]:
#@title ▶ Download All Receptor Files
#@markdown Compresses all generated receptor map files into a single `.tar.gz` archive
#@markdown and initiates a download to your local computer.

import os, subprocess
from google.colab import files

archive_name = f'{receptor_name}_maps.tar.gz'
archive_path = f'/content/{archive_name}'

# List what will be packaged
all_files = os.listdir(work_dir)
relevant  = [f for f in all_files
             if not f.startswith('.') and os.path.isfile(os.path.join(work_dir, f))]
print(f'📦 Packaging {len(relevant)} files:')
for f in sorted(relevant):
    kb = os.path.getsize(os.path.join(work_dir, f)) / 1024
    print(f'   {f:45s} {kb:8.1f} KB')

# Create archive
os.system(f'tar -czvf {archive_path} -C {work_dir} ' +
          ' '.join(f"'{f}'" for f in sorted(relevant)))

if os.path.exists(archive_path):
    size_mb = os.path.getsize(archive_path) / (1024*1024)
    print(f'\n✅ Archive created: {archive_path}  ({size_mb:.1f} MB)')
    print('⬇️  Starting download ...')
    files.download(archive_path)
else:
    print('❌ Archive creation failed.')

print('\n🎉 Receptor preparation pipeline complete!')
print(f'   Extract {archive_name} and place contents in:')
print(f'   docking/rigid/<your_receptor_name>/')
if globals().get('flexible_residues_defined', False):
    print(f'\n🦴 Flexible residues were defined in Step 3.5:')
    print(f'   {receptor_name}.pdbqt       → rigid part (--receptor / -r at docking time)')
    print(f'   {receptor_name}_flex.pdbqt  → flexible part (--flexres / -x at docking time)')
else:
    print(f'\nℹ️  No flexible residues were defined — {receptor_name}.pdbqt is a standard rigid receptor.')
